# Which culture medium makes a neuron grow its axon? A FluxQuery analysis

**What this notebook does.** We take a genome-scale metabolic model of a human neuron (here the dopaminergic model iDopaNeuroC) together with a large sample of its feasible flux states, and we ask one question: *what does the metabolism of a neuron that is actively growing its axon look like, and which medium would push the cell towards that state?*

The question is written down, reaction by reaction, in a Word document, the **query**. Requirements are relative: "the activity of this reaction should be in the upper part of what the model can do". The notebook then:

1. checks the query against the model, then removes thermodynamically infeasible cycles from the flux sampling (they turned out to dominate it);
2. trains two autoencoders on the complete, loop-free flux space: a plain one (AE) and a physics-informed one (PI-AE);
3. searches the learned space for growth states (**FluxQuery**) and turns every candidate into an exact steady state of the model;
4. compares the result with what classical single objectives (maximise ATP demand, maximise lipid synthesis, …) would predict;
5. translates the growth states into nutrient demand and, by linear programming, into a **culture medium**.

Everything lives in this notebook: the analysis and, in a hidden *Toolkit* section, every function it uses. The neural search builds on the original FluxQuery idea: learn the complete feasible flux space first, and only afterwards ask it about the phenotype. The validation run uses 8,000 samples (`smallSamples.mat`); the production run uses the full sampling (`samples1-3.mat`). The same notebook and the same query serve the glutamatergic and cholinergic projects: only `CONTEXT`, the model and its sampling change.

## 0. Settings

**Requirements.** Python 3.9 or newer and `pip install numpy scipy pandas scikit-learn matplotlib openpyxl python-docx pyyaml` (add `h5py` for MATLAB v7.3 files). No GPU or deep-learning framework is needed.

**Inputs.** Leave a path as `None` and the notebook asks for it with a dialog (or a typed path when no display is available):

| Setting | What to select |
|---|---|
| `MODEL_FILE` | the COBRA model, `.mat` (e.g. `iDopaNeuroC.mat`) |
| `SAMPLING_DIR` | the **folder** with the flux samplings (`samples1.mat`, `samples2.mat`, … or `smallSamples.mat` alone for a pilot) |
| `QUERY_FILE` | the query, Word `.docx` (`axonal_growth_query_v4.docx`) |
| `BIBLIOMIC_FILE` | the bibliomic workbook with the `mediaData` and `cellCultureData` sheets |
| `OUTPUT_DIR` | where figures, tables and the medium recommendation are written |

The first time a sampling folder is used, every `.mat` in it is converted to a NumPy `.npy` file in a `python/` subfolder. Later runs read the `.npy` files directly (memory-mapped), so only the states actually used are loaded. Paths can also be given through the environment variables `FQ_MODEL`, `FQ_SAMPLES`, `FQ_QUERY`, `FQ_BIBLIOMIC` and `FQ_OUTPUT`, which is convenient for unattended runs.

In [ ]:
CONTEXT = "dopaminergic"        # "dopaminergic", "glutamatergic" or "cholinergic" (must match the query's Context column)

MODEL_FILE = None               # COBRA model, MATLAB .mat (e.g. iDopaNeuroC.mat)
SAMPLING_DIR = None             # folder with the sampling files (.mat); all samplings in it are pooled
QUERY_FILE = None               # the query: Word .docx (or .xlsx / .csv / .yaml)
BIBLIOMIC_FILE = None           # bibliomic workbook with the mediaData and cellCultureData sheets
OUTPUT_DIR = None               # where figures, tables and the medium recommendation are written

MAX_SAMPLES = 30000             # states drawn at random from the pooled samplings; None keeps everything
LOOP_FREE = True                # remove thermodynamically infeasible cycles (CycleFreeFlux) before learning

# Autoencoders (original FluxQuery architecture: n -> 512 -> 128 -> latent -> 128 -> 512 -> n)
LATENT_DIM, EPOCHS, BATCH_SIZE, LEARNING_RATE = 256, 80, 256, 3e-3
PI_LAMBDA_MASS, PI_LAMBDA_BOUNDS = 1e-6, 1e-3

# FluxQuery latent search
N_SEEDS, FQ_STEPS, FQ_LR, FQ_LAMBDA_BIO, FQ_LAMBDA_LATENT = 32, 400, 2e-2, 10.0, 0.05

# Medium design
MAX_FOLD = 10                   # no component may exceed 10x its base concentration
SUPPLEMENT_DOSE_UM = 100        # test dose for compounds absent from the base medium
SEED = 42

## Toolkit

The functions used by the analysis. The code cells are collapsed: run them, and open them only if you want to read or change the method. Each cell is self-contained and documented.

In [ ]:
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, List, Optional, Sequence
import re
from typing import Dict, List, Optional
from typing import Tuple
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import time
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional
from typing import Dict, Optional
from typing import Dict, Iterable, List, Optional, Sequence
from typing import Dict, List, Optional, Sequence

import numpy as np
import scipy.io as sio
import scipy.sparse as sp
import pandas as pd
from scipy.optimize import linprog
from sklearn.neighbors import NearestNeighbors
import matplotlib as mpl
import matplotlib.pyplot as plt

import json
import time
import warnings
import pandas as pd
import matplotlib.pyplot as plt

VERSION = "4.1.0"
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Files, model and samplings

Input/output helpers: file selection, COBRA model loading and flux-sampling loading.

In [ ]:
# ---------------------------------------------------------------------------
# File selection
# ---------------------------------------------------------------------------

def select_file(title: str, patterns: Sequence[str] = ("*",), multiple: bool = False,
                env_var: Optional[str] = None):
    """Ask the user for one (or several) files and return Path / list of Path.

    Resolution order:
      1. environment variable ``env_var`` (useful for batch runs / CI);
      2. a native file dialog (tkinter), if a display is available;
      3. a typed path at the prompt (works in any terminal or notebook).
    Several paths can be typed separated by ';'.
    """
    if env_var and os.environ.get(env_var):
        paths = [Path(p).expanduser() for p in os.environ[env_var].split(";") if p.strip()]
        return paths if multiple else paths[0]

    try:  # native dialog
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        ftypes = [(title, " ".join(patterns)), ("All files", "*")]
        if multiple:
            sel = filedialog.askopenfilenames(title=title, filetypes=ftypes)
            root.destroy()
            if sel:
                return [Path(p) for p in sel]
        else:
            sel = filedialog.askopenfilename(title=title, filetypes=ftypes)
            root.destroy()
            if sel:
                return Path(sel)
    except Exception:
        pass

    hint = " (separate several files with ';')" if multiple else ""
    raw = input(f"{title} {list(patterns)}{hint}: ").strip().strip('"').strip("'")
    paths = [Path(p.strip().strip('"')).expanduser() for p in raw.split(";") if p.strip()]
    if not paths:
        raise FileNotFoundError(f"No file given for: {title}")
    for p in paths:
        if not p.exists():
            raise FileNotFoundError(p)
    return paths if multiple else paths[0]


def select_folder(title: str, env_var: Optional[str] = None, create: bool = True) -> Path:
    """Ask for an output folder (dialog or prompt)."""
    if env_var and os.environ.get(env_var):
        p = Path(os.environ[env_var]).expanduser()
    else:
        p = None
        try:
            import tkinter as tk
            from tkinter import filedialog
            root = tk.Tk()
            root.withdraw()
            root.attributes("-topmost", True)
            sel = filedialog.askdirectory(title=title)
            root.destroy()
            p = Path(sel) if sel else None
        except Exception:
            pass
        if p is None:
            p = Path(input(f"{title}: ").strip().strip('"')).expanduser()
    if create:
        p.mkdir(parents=True, exist_ok=True)
    return p


# ---------------------------------------------------------------------------
# COBRA model
# ---------------------------------------------------------------------------

@dataclass
class Model:
    """Minimal read-only view of a COBRA model."""
    name: str
    S: sp.csr_matrix
    rxns: List[str]
    mets: List[str]
    lb: np.ndarray
    ub: np.ndarray
    rxn_names: List[str] = field(default_factory=list)
    formulas: List[str] = field(default_factory=list)
    subsystems: List[str] = field(default_factory=list)
    met_names: List[str] = field(default_factory=list)

    @property
    def n(self) -> int:
        return len(self.rxns)

    def index(self, rxn: str) -> int:
        if not hasattr(self, "_idx"):
            self._idx = {r: i for i, r in enumerate(self.rxns)}
        return self._idx[rxn]

    def has(self, rxn: str) -> bool:
        if not hasattr(self, "_idx"):
            self._idx = {r: i for i, r in enumerate(self.rxns)}
        return rxn in self._idx

    def exchanges(self) -> List[int]:
        return [i for i, r in enumerate(self.rxns) if r.startswith("EX_")]

    def copy_bounds(self):
        return self.lb.copy(), self.ub.copy()


def _str_list(x) -> List[str]:
    if x is None:
        return []
    arr = np.atleast_1d(x)
    out = []
    for v in arr.ravel():
        if isinstance(v, np.ndarray):
            v = v.ravel()[0] if v.size else ""
        out.append(str(v))
    return out


def _subsystems(x, n) -> List[str]:
    out = []
    for v in np.atleast_1d(x).ravel():
        if isinstance(v, np.ndarray):
            v = "; ".join(str(s) for s in v.ravel())
        out.append(str(v))
    return out if len(out) == n else [""] * n


def load_model(path, variable: Optional[str] = None) -> Model:
    """Load a COBRA model saved by MATLAB (v5/v7; v7.3 needs to be re-saved with -v7)."""
    path = Path(path)
    try:
        raw = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
    except NotImplementedError as exc:  # HDF5 / v7.3
        raise RuntimeError(f"{path.name} is a MATLAB v7.3 file. Re-save it in MATLAB with "
                           "save(file, 'model', '-v7').") from exc
    keys = [k for k in raw if not k.startswith("__")]
    if variable is None:
        cands = [k for k in keys if hasattr(raw[k], "_fieldnames")
                 and {"S", "rxns", "lb", "ub"} <= set(raw[k]._fieldnames)]
        if not cands:
            raise ValueError(f"No COBRA structure (S, rxns, lb, ub) found in {path.name}; keys={keys}")
        variable = cands[0]
    m = raw[variable]
    S = sp.csr_matrix(m.S, dtype=float)
    rxns = _str_list(m.rxns)
    fields = set(m._fieldnames)
    get = lambda f: getattr(m, f) if f in fields else None
    name = str(get("modelID") or get("description") or variable)
    return Model(
        name=variable if name in ("", "None") else name,
        S=S, rxns=rxns, mets=_str_list(m.mets),
        lb=np.asarray(m.lb, float).ravel(), ub=np.asarray(m.ub, float).ravel(),
        rxn_names=_str_list(get("rxnNames")) or [""] * len(rxns),
        formulas=_str_list(get("rxnFormulas")) or [""] * len(rxns),
        subsystems=_subsystems(get("subSystems"), len(rxns)) if get("subSystems") is not None else [""] * len(rxns),
        met_names=_str_list(get("metNames")),
    )


# ---------------------------------------------------------------------------
# Flux sampling
# ---------------------------------------------------------------------------

def _read_one_sampling(path: Path, n_rxns: int, variable: Optional[str]) -> np.ndarray:
    suffix = path.suffix.lower()
    if suffix == ".mat":
        try:
            raw = sio.loadmat(path)
        except NotImplementedError:
            import h5py  # optional dependency for v7.3 files
            with h5py.File(path, "r") as f:
                key = variable or [k for k in f.keys() if not k.startswith("#")][0]
                X = np.asarray(f[key])
        else:
            keys = [k for k in raw if not k.startswith("__")]
            if variable and variable in raw:
                key = variable
            else:  # first 2-D numeric array with one dimension equal to n_rxns
                key = next((k for k in keys if isinstance(raw[k], np.ndarray) and raw[k].ndim == 2
                            and n_rxns in raw[k].shape and raw[k].dtype.kind in "fiu"), None)
                if key is None:
                    raise ValueError(f"No (n_reactions x n_samples) matrix in {path.name}; keys={keys}")
            X = raw[key]
    elif suffix in (".csv", ".txt", ".tsv"):
        X = np.loadtxt(path, delimiter="," if suffix == ".csv" else None)
    elif suffix == ".npy":
        X = np.load(path)
    else:
        raise ValueError(f"Unsupported sampling format: {path}")
    X = np.asarray(X)
    if X.shape[0] == n_rxns and X.shape[1] != n_rxns:
        X = X.T
    if X.shape[1] != n_rxns:
        raise ValueError(f"{path.name}: shape {X.shape} does not match {n_rxns} model reactions")
    return X.astype(np.float32, copy=False)


def load_samples(paths, model: Model, variable: Optional[str] = None,
                 max_samples: Optional[int] = None, seed: int = 0) -> np.ndarray:
    """Load one or several sampling files and return an (n_samples x n_reactions) float32 array.

    ``max_samples`` randomly thins the pooled sampling (without replacement) to bound memory
    and training time; chains are pooled before thinning so every file contributes.
    """
    if isinstance(paths, (str, Path)):
        paths = [paths]
    rng = np.random.default_rng(seed)
    blocks = []
    per_file = None if max_samples is None else int(np.ceil(max_samples / len(paths)))
    for p in paths:
        X = _read_one_sampling(Path(p), model.n, variable)
        if per_file is not None and len(X) > per_file:
            X = X[np.sort(rng.choice(len(X), per_file, replace=False))]
        blocks.append(X)
    X = np.vstack(blocks)
    if max_samples is not None and len(X) > max_samples:
        X = X[np.sort(rng.choice(len(X), max_samples, replace=False))]
    return X


def convert_samplings(folder, model: Model, variable: Optional[str] = None, out_subdir: str = "python",
                      log=print) -> List[Path]:
    """Convert every MATLAB sampling file in `folder` into a NumPy ``.npy`` file.

    Each ``<name>.mat`` becomes ``<folder>/<out_subdir>/<name>.npy``: a float32 matrix of
    (samples x reactions) in the reaction order of `model`, plus a ``reactions.txt`` with that
    order. ``.npy`` files open instantly and can be memory-mapped, so later runs read only the
    rows they need instead of decompressing gigabytes of ``.mat``. A file is converted again
    only when its ``.mat`` is newer than its ``.npy``. ``.mat`` files that do not hold a
    sampling of this model (for example the model itself) are skipped.
    """
    folder = Path(folder)
    out = folder / out_subdir
    out.mkdir(exist_ok=True)
    (out / "reactions.txt").write_text("\n".join(model.rxns) + "\n")
    converted = []
    for mat in sorted(folder.glob("*.mat")):
        npy = out / (mat.stem + ".npy")
        if npy.exists() and npy.stat().st_mtime >= mat.stat().st_mtime:
            log(f"  {mat.name}: already converted")
            converted.append(npy)
            continue
        try:
            X = _read_one_sampling(mat, model.n, variable)
        except (ValueError, KeyError, StopIteration) as err:
            log(f"  {mat.name}: skipped ({err})")
            continue
        np.save(npy, np.ascontiguousarray(X, dtype=np.float32))
        log(f"  {mat.name}: {X.shape[0]:,} states -> {npy.relative_to(folder)} ({npy.stat().st_size / 1e6:,.0f} MB)")
        converted.append(npy)
        del X
    if not converted:
        raise FileNotFoundError(f"No sampling of a {model.n}-reaction model found in {folder}")
    return converted


def load_converted(npy_paths: Sequence[Path], max_samples: Optional[int] = None, seed: int = 0) -> np.ndarray:
    """Pool converted samplings (memory-mapped) and draw at most `max_samples` states at random,
    reading only the selected rows from disk."""
    arrays = [np.load(p, mmap_mode="r") for p in npy_paths]
    sizes = np.array([len(a) for a in arrays])
    total = int(sizes.sum())
    rng = np.random.default_rng(seed)
    take = np.arange(total) if max_samples is None or max_samples >= total else \
        np.sort(rng.choice(total, max_samples, replace=False))
    starts = np.concatenate([[0], np.cumsum(sizes)])
    blocks = []
    for a, s0, s1 in zip(arrays, starts[:-1], starts[1:]):
        rows = take[(take >= s0) & (take < s1)] - s0
        if len(rows):
            blocks.append(np.asarray(a[rows], dtype=np.float32))
    return np.vstack(blocks)


def sampling_validation(X: np.ndarray, model: Model, tol: float = 1e-6) -> dict:
    """Check that the sampled points are steady states within the model bounds."""
    res = np.abs(model.S @ X.T.astype(float))            # m x N
    flux_scale = np.maximum(np.abs(X).max(axis=1), 1.0)
    rel = res.max(axis=0) / flux_scale
    low = (model.lb[None, :] - X) > tol * np.maximum(1, np.abs(model.lb))[None, :]
    up = (X - model.ub[None, :]) > tol * np.maximum(1, np.abs(model.ub))[None, :]
    return {
        "n_samples": int(X.shape[0]),
        "n_reactions": int(X.shape[1]),
        "max_abs_Sv": float(res.max()),
        "median_relative_Sv": float(np.median(rel)),
        "fraction_samples_with_bound_violation": float(np.mean((low | up).any(axis=1))),
        "finite": bool(np.isfinite(X).all()),
    }

### The query

The biological query: reading, context selection, validation and percentile thresholds.

In [ ]:
COLUMNS = ["context", "module", "process", "reaction", "direction", "p_low", "p_high",
           "status", "group", "rationale", "reference"]
DIRECTIONS = {"forward": "forward", "reverse": "reverse", "magnitude": "magnitude",
              "uptake": "reverse", "secretion": "forward", "abs": "magnitude"}
STATUSES = {"active", "diagnostic"}


# ---------------------------------------------------------------------------
# Reading
# ---------------------------------------------------------------------------

def _normalise(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"[^a-z_]", "", str(c).strip().lower().replace(" ", "_")) for c in df.columns]
    aliases = {"reaction_id": "reaction", "rxn": "reaction", "biological_direction": "direction",
               "plow": "p_low", "phigh": "p_high", "constraint_group": "group", "note": "rationale"}
    df = df.rename(columns={k: v for k, v in aliases.items() if k in df.columns})
    for c in COLUMNS:
        if c not in df.columns:
            df[c] = np.nan
    df = df[COLUMNS]
    df = df[df["reaction"].astype(str).str.strip().ne("") & df["reaction"].notna()]
    df["reaction"] = df["reaction"].astype(str).str.strip()
    df["context"] = df["context"].fillna("all").astype(str).str.strip().str.lower()
    df["direction"] = df["direction"].fillna("forward").astype(str).str.strip().str.lower().map(
        lambda d: DIRECTIONS.get(d, d))
    bad = set(df["direction"]) - set(DIRECTIONS.values())
    if bad:
        raise ValueError(f"Unknown direction(s) in query: {bad}")
    for c in ("p_low", "p_high"):
        df[c] = pd.to_numeric(df[c].replace({"-": np.nan, "": np.nan, "—": np.nan}), errors="coerce")
    df["status"] = df["status"].fillna("active").astype(str).str.strip().str.lower()
    df.loc[~df["status"].isin(STATUSES), "status"] = "diagnostic"
    df["group"] = df["group"].where(df["group"].notna() & df["group"].astype(str).str.strip().ne(""), df["reaction"])
    for c in ("module", "process", "rationale", "reference"):
        df[c] = df[c].fillna("").astype(str).str.strip()
    return df.reset_index(drop=True)


def _read_docx(path: Path) -> pd.DataFrame:
    try:
        import docx  # python-docx
    except ImportError as exc:
        raise ImportError("Reading a Word query requires python-docx (pip install python-docx).") from exc
    doc = docx.Document(str(path))
    for table in doc.tables:
        rows = [[c.text.strip() for c in r.cells] for r in table.rows]
        header = [h.lower() for h in rows[0]]
        if any("reaction" in h for h in header) and any("direction" in h for h in header):
            return pd.DataFrame(rows[1:], columns=rows[0])
    raise ValueError(f"{path.name}: no requirement table with 'Reaction' and 'Direction' columns was found.")


def _read_yaml(path: Path) -> pd.DataFrame:
    import yaml
    data = yaml.safe_load(open(path, encoding="utf-8"))
    rows = []
    for block in data.get("requirements", []):
        for r in block.get("reactions", []):
            status = str(r.get("status", "active")).lower()
            rows.append({
                "context": block.get("context", r.get("context", "all")),
                "module": block.get("module", ""), "process": block.get("process", ""),
                "reaction": r.get("model_rxn") or r.get("source_rxn"),
                "direction": r.get("biological_direction", "forward"),
                "p_low": r.get("p_low"), "p_high": r.get("p_high"),
                "status": "active" if status == "active" and r.get("use_for_score", True) else "diagnostic",
                "group": r.get("constraint_group"), "rationale": r.get("note", ""),
                "reference": block.get("reference_index", ""),
            })
    return pd.DataFrame(rows)


def read_query(path) -> pd.DataFrame:
    """Read a query file (.docx, .xlsx, .csv or .yaml) into the canonical table."""
    path = Path(path)
    suf = path.suffix.lower()
    if suf == ".docx":
        df = _read_docx(path)
    elif suf in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    elif suf == ".csv":
        df = pd.read_csv(path)
    elif suf in (".yaml", ".yml"):
        df = _read_yaml(path)
    else:
        raise ValueError(f"Unsupported query format: {path.suffix}")
    return _normalise(df)


def select_context(query: pd.DataFrame, context: str) -> pd.DataFrame:
    """Keep rows whose Context is 'all' or contains the requested context."""
    context = context.strip().lower()
    keep = query["context"].apply(lambda c: c == "all" or context in [x.strip() for x in re.split(r"[,;/]", c)])
    return query.loc[keep].reset_index(drop=True)


def available_contexts(query: pd.DataFrame) -> List[str]:
    out = set()
    for c in query["context"]:
        out.update(x.strip() for x in re.split(r"[,;/]", c) if x.strip() and x.strip() != "all")
    return sorted(out)


# ---------------------------------------------------------------------------
# Activity transform and thresholds
# ---------------------------------------------------------------------------

def activity(V: np.ndarray, direction) -> np.ndarray:
    """Map signed model fluxes to a biologically oriented activity (column-wise)."""
    V = np.asarray(V, float)
    direction = np.atleast_1d(direction)
    A = V.copy()
    rev = direction == "reverse"
    mag = direction == "magnitude"
    if A.ndim == 1:
        A = A[None, :]
    A[:, rev] = -A[:, rev]
    A[:, mag] = np.abs(A[:, mag])
    return A


def build_constraints(query: pd.DataFrame, model: Model, X: np.ndarray) -> pd.DataFrame:
    """Translate percentile requirements into model-specific numeric targets.

    Thresholds are recomputed from *this* model's sampling, so the same query can be
    reused across GEMs. The normalisation scale is the inter-quartile range of the
    activity (minimum 1e-6) so violations are comparable across reactions.
    """
    q = query[query["reaction"].map(model.has)].copy()
    j = q["reaction"].map(model.index).to_numpy(int)
    A = activity(X[:, j], q["direction"].to_numpy())
    p_lo, p_hi = q["p_low"].to_numpy(float), q["p_high"].to_numpy(float)
    q["rxn_index"] = j
    q["target_lb"] = [np.percentile(A[:, k], p) if np.isfinite(p) else np.nan for k, p in enumerate(p_lo)]
    q["target_ub"] = [np.percentile(A[:, k], p) if np.isfinite(p) else np.nan for k, p in enumerate(p_hi)]
    q25, q50, q75 = np.percentile(A, [25, 50, 75], axis=0)
    q["activity_median"] = q50
    q["scale"] = np.maximum(q75 - q25, 1e-6)
    return q.reset_index(drop=True)


# ---------------------------------------------------------------------------
# Validation of a query against a model and its sampling
# ---------------------------------------------------------------------------

def validate_query(query: pd.DataFrame, model: Model, X: np.ndarray,
                   open_bound: float = 1e4, loop_fraction: float = 0.5, coupling_r: float = 0.999) -> pd.DataFrame:
    """Biological and numerical sanity checks for every requirement row.

    Flags
    -----
    missing          reaction ID not in the model
    blocked          activity is (almost) constant in the sampling -> no discrimination
    wrong_sign       the requested activity is negative in >50 % of samples
    extreme          the P95 |activity| reaches >= loop_fraction of the generic open bound (1e4):
                     flux of that size is carried by thermodynamically infeasible cycles or by
                     artefactual (non-medium) uptakes, so its percentiles do not rank biology
    coupled_with     reactions whose sampled activities are perfectly correlated
                     (should share one Group so they count once)
    """
    rows = []
    present = query["reaction"].map(model.has).to_numpy()
    idx = [model.index(r) if ok else -1 for r, ok in zip(query["reaction"], present)]
    for k, r in query.iterrows():
        d = {"reaction": r["reaction"], "module": r["module"], "direction": r["direction"],
             "status": r["status"], "in_model": bool(present[k])}
        if present[k]:
            j = idx[k]
            a = activity(X[:, j][:, None], np.array([r["direction"]]))[:, 0]
            p = np.percentile(a, [5, 25, 50, 75, 95])
            d.update({"lb": model.lb[j], "ub": model.ub[j], "P5": p[0], "P50": p[2], "P95": p[4],
                      "frac_positive_activity": float(np.mean(a > 1e-9)),
                      "blocked": bool((p[4] - p[0]) < 1e-4 or np.max(np.abs(a)) < 1e-4),
                      "wrong_sign": bool(np.mean(a > 1e-9) < 0.5),
                      "extreme": bool(np.percentile(np.abs(a), 95) >= loop_fraction * open_bound)})
        rows.append(d)
    out = pd.DataFrame(rows)
    # perfect coupling among mapped requirements
    ok = [k for k in range(len(query)) if present[k]]
    if len(ok) > 1:
        A = activity(X[:, [idx[k] for k in ok]], query.loc[ok, "direction"].to_numpy())
        sd = A.std(axis=0)
        C = np.corrcoef(np.where(sd[None, :] > 0, A, A + 1e-12 * np.random.default_rng(0).standard_normal(A.shape)).T)
        partners = {k: [] for k in ok}
        for a_i, ka in enumerate(ok):
            for b_i, kb in enumerate(ok):
                if a_i < b_i and abs(C[a_i, b_i]) >= coupling_r and sd[a_i] > 0 and sd[b_i] > 0:
                    partners[ka].append(query.loc[kb, "reaction"])
                    partners[kb].append(query.loc[ka, "reaction"])
        out["coupled_with"] = [", ".join(partners.get(k, [])) for k in range(len(query))]
    else:
        out["coupled_with"] = ""

    def verdict(r):
        if not r["in_model"]:
            return "not in model -> ignored"
        if r["blocked"]:
            return "blocked/near-fixed -> keep as diagnostic"
        if r["wrong_sign"]:
            return "activity mostly opposite to the requested direction -> check direction"
        if r["extreme"]:
            return "extreme flux (near the generic bound) -> prefer a bounded anchor; keep as diagnostic"
        return "ok"
    out["verdict"] = out.apply(verdict, axis=1)
    return out


def apply_validation(query: pd.DataFrame, report: pd.DataFrame) -> pd.DataFrame:
    """Demote rows that cannot be used for scoring (missing, blocked, wrong sign, extreme flux)
    to 'diagnostic', and merge perfectly coupled active rows into one group. Returns a copy."""
    q = query.copy()
    bad = report["verdict"].ne("ok").to_numpy()
    q.loc[bad & q["status"].eq("active").to_numpy(), "status"] = "diagnostic"
    for k, r in report.iterrows():
        if r.get("coupled_with") and q.loc[k, "status"] == "active" and q.loc[k, "group"] == q.loc[k, "reaction"]:
            members = sorted([q.loc[k, "reaction"]] + [x.strip() for x in r["coupled_with"].split(",")])
            q.loc[k, "group"] = "+".join(members)
    return q

### Scoring flux states against the query

Biological compliance of flux vectors with respect to the query constraints.

In [ ]:
def active(cons: pd.DataFrame) -> pd.DataFrame:
    return cons[cons["status"].eq("active")].reset_index(drop=True)


def _violations(V: np.ndarray, cons: pd.DataFrame) -> np.ndarray:
    V = np.atleast_2d(V)
    A = activity(V[:, cons["rxn_index"].to_numpy(int)], cons["direction"].to_numpy())
    lb, ub = cons["target_lb"].to_numpy(float), cons["target_ub"].to_numpy(float)
    low = np.where(np.isfinite(lb), np.maximum(np.nan_to_num(lb) - A, 0.0), 0.0)
    up = np.where(np.isfinite(ub), np.maximum(A - np.nan_to_num(ub), 0.0), 0.0)
    return (low + up) / cons["scale"].to_numpy(float)


def group_matrix(cons: pd.DataFrame) -> Tuple[np.ndarray, list]:
    groups = list(dict.fromkeys(cons["group"].astype(str)))
    G = np.zeros((len(cons), len(groups)))
    for k, g in enumerate(cons["group"].astype(str)):
        G[k, groups.index(g)] = 1.0
    return G / G.sum(axis=0, keepdims=True), groups


def compliance(V: np.ndarray, cons: pd.DataFrame, tol: float = 1e-9):
    """Return (compliance_fraction, mean_violation, soft_score) per flux vector."""
    act = active(cons)
    E = _violations(V, act)
    G, _ = group_matrix(act)
    gviol = (E ** 2) @ G
    gok = ((E <= tol).astype(float) @ (G > 0)) >= (G > 0).sum(axis=0) - 1e-9
    frac = gok.mean(axis=1)
    mv = gviol.mean(axis=1)
    return frac, mv, 1.0 / (1.0 + mv)


def module_compliance(V: np.ndarray, cons: pd.DataFrame) -> pd.DataFrame:
    """Compliance fraction per module (rows = vectors, columns = modules)."""
    act = active(cons)
    out = {}
    for mod in dict.fromkeys(act["module"]):
        sub = act[act["module"] == mod].reset_index(drop=True)
        out[mod] = compliance(V, sub)[0]
    return pd.DataFrame(out)


def row_report(v: np.ndarray, cons: pd.DataFrame) -> pd.DataFrame:
    """Per-requirement table for a single flux vector."""
    v = np.asarray(v, float).ravel()
    A = activity(v[cons["rxn_index"].to_numpy(int)][None, :], cons["direction"].to_numpy())[0]
    e = _violations(v[None, :], cons)[0]
    out = cons[["module", "reaction", "direction", "status", "target_lb", "target_ub"]].copy()
    out["activity"] = A
    out["normalised_violation"] = e
    out["met"] = e <= 1e-9
    return out


def bio_penalty_and_grad(V: np.ndarray, cons: pd.DataFrame, n_rxns: int):
    """Grouped squared penalty (FluxQuery objective) and its gradient w.r.t. the fluxes V (B x n)."""
    act = active(cons)
    j = act["rxn_index"].to_numpy(int)
    d = act["direction"].to_numpy()
    sign = np.where(d == "reverse", -1.0, 1.0)
    Vj = V[:, j]
    A = Vj * sign
    mag = d == "magnitude"
    A[:, mag] = np.abs(Vj[:, mag])
    dA = np.broadcast_to(sign, A.shape).copy()
    dA[:, mag] = np.sign(Vj[:, mag])
    lb, ub = act["target_lb"].to_numpy(float), act["target_ub"].to_numpy(float)
    s = act["scale"].to_numpy(float)
    low = np.where(np.isfinite(lb), np.maximum(np.nan_to_num(lb) - A, 0.0), 0.0) / s
    up = np.where(np.isfinite(ub), np.maximum(A - np.nan_to_num(ub), 0.0), 0.0) / s
    e = low + up
    G, groups = group_matrix(act)
    gviol = (e ** 2) @ G                       # B x g
    loss = gviol.mean()
    # d loss / d e_k = (1/(B*g)) * 2 e_k * G[k, group(k)]
    w = G.sum(axis=1)                           # weight of row k inside its group
    dE = 2.0 * e * w[None, :] / (V.shape[0] * len(groups))
    dA_total = dE * (np.where(up > 0, 1.0, 0.0) - np.where(low > 0, 1.0, 0.0)) / s
    grad = np.zeros_like(V)
    np.add.at(grad.T, j, (dA_total * dA).T)
    return float(loss), grad

### Linear programs

Linear programs on the GEM (scipy/HiGHS; no commercial solver required).

In [ ]:
OPTS = {"presolve": True, "primal_feasibility_tolerance": 1e-8, "dual_feasibility_tolerance": 1e-8}


class LPError(RuntimeError):
    pass


def _solve(c, A_ub, b_ub, A_eq, b_eq, bounds):
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs", options=OPTS)
    if res.status != 0:
        raise LPError(f"LP failed (status {res.status}): {res.message}")
    return res


def _bounds(lb, ub):
    return np.column_stack([lb, ub])


def flux_scale(X: np.ndarray) -> np.ndarray:
    """Per-reaction scale used for distances (robust spread of the sampling, >= 1)."""
    q05, q95 = np.percentile(X, [5, 95], axis=0)
    return np.maximum(q95 - q05, 1.0)


# ---------------------------------------------------------------------------
# Biological rows as linear inequalities
# ---------------------------------------------------------------------------

def _bio_rows(cons: pd.DataFrame, n: int, X_sign: Optional[np.ndarray] = None):
    """Return (sign per row, row weights, act table). Magnitude rows are linearised with the
    dominant sampled sign of the reaction (documented approximation; no MILP needed)."""
    act = active(cons)
    d = act["direction"].to_numpy()
    sign = np.where(d == "reverse", -1.0, 1.0)
    if X_sign is not None:
        mag = d == "magnitude"
        sign[mag] = np.where(X_sign[act.loc[mag, "rxn_index"].to_numpy(int)] < 0, -1.0, 1.0)
    G, _ = group_matrix(act)
    w = G.sum(axis=1) / act["scale"].to_numpy(float)       # group-balanced, scale-normalised
    return act, sign, w


def query_lp(model: Model, cons: pd.DataFrame, lb=None, ub=None, v0: Optional[np.ndarray] = None,
             scale: Optional[np.ndarray] = None, rho: float = 1.0, median_sign: Optional[np.ndarray] = None,
             extra_ub: Optional[Tuple[sp.spmatrix, np.ndarray]] = None) -> Dict:
    """min  rho * sum_k w_k (s_k^- + s_k^+)  [+ sum_i |v_i - v0_i| / scale_i  if v0 is given]

    s.t. S v = 0, lb <= v <= ub, a_k(v) + s_k^- >= target_lb_k, a_k(v) - s_k^+ <= target_ub_k.
    With v0=None this is the exact-feasible biological benchmark (the best compliance the
    GEM can reach); with v0 it is the biology-aware repair of a neural candidate.
    """
    n = model.n
    lb = model.lb if lb is None else lb
    ub = model.ub if ub is None else ub
    act, sign, w = _bio_rows(cons, n, median_sign)
    k = len(act)
    j = act["rxn_index"].to_numpy(int)
    tl, tu = act["target_lb"].to_numpy(float), act["target_ub"].to_numpy(float)
    has_l, has_u = np.isfinite(tl), np.isfinite(tu)
    use_d = v0 is not None
    nv = n + 2 * k + (n if use_d else 0)
    c = np.zeros(nv)
    c[n:n + k] = rho * w
    c[n + k:n + 2 * k] = rho * w
    rows, rhs = [], []
    # -(sign*v_j) - s^- <= -tl
    L = sp.lil_matrix((k, nv))
    for r in range(k):
        if has_l[r]:
            L[r, j[r]] = -sign[r]
            L[r, n + r] = -1.0
    U = sp.lil_matrix((k, nv))
    for r in range(k):
        if has_u[r]:
            U[r, j[r]] = sign[r]
            U[r, n + k + r] = -1.0
    rows += [L.tocsr()[has_l], U.tocsr()[has_u]]
    rhs += [-tl[has_l], tu[has_u]]
    bnds = [_bounds(lb, ub), np.tile([0.0, np.inf], (2 * k, 1))]
    if use_d:
        scale = np.ones(n) if scale is None else scale
        c[n + 2 * k:] = 1.0 / scale
        I = sp.eye(n, format="csr")
        Z = sp.csr_matrix((n, 2 * k))
        rows += [sp.hstack([I, Z, -I]), sp.hstack([-I, Z, -I])]
        rhs += [v0, -v0]
        bnds.append(np.tile([0.0, np.inf], (n, 1)))
    if extra_ub is not None:
        A_x, b_x = extra_ub
        rows.append(sp.hstack([A_x, sp.csr_matrix((A_x.shape[0], nv - n))]))
        rhs.append(b_x)
    A_eq = sp.hstack([model.S, sp.csr_matrix((model.S.shape[0], nv - n))]).tocsr()
    res = _solve(c, sp.vstack(rows).tocsr(), np.concatenate(rhs), A_eq, np.zeros(model.S.shape[0]),
                 np.vstack(bnds))
    v = res.x[:n]
    return {"v": v, "objective": res.fun, "bio_violation": float(np.sum(w * (res.x[n:n + k] + res.x[n + k:n + 2 * k]))),
            "distance": float(np.sum(np.abs(v - v0) / scale)) if use_d else np.nan}


def nearest_feasible(model: Model, v0: np.ndarray, scale: np.ndarray, lb=None, ub=None) -> np.ndarray:
    """Exact repair of the original FluxQuery code: nearest steady state under normalised L1."""
    n = model.n
    lb = model.lb if lb is None else lb
    ub = model.ub if ub is None else ub
    I = sp.eye(n, format="csr")
    A_ub = sp.vstack([sp.hstack([I, -I]), sp.hstack([-I, -I])]).tocsr()
    c = np.concatenate([np.zeros(n), 1.0 / scale])
    A_eq = sp.hstack([model.S, sp.csr_matrix(model.S.shape)]).tocsr()
    res = _solve(c, A_ub, np.concatenate([v0, -v0]), A_eq, np.zeros(model.S.shape[0]),
                 np.vstack([_bounds(lb, ub), np.tile([0.0, np.inf], (n, 1))]))
    return res.x[:n]


# ---------------------------------------------------------------------------
# Classical objective functions
# ---------------------------------------------------------------------------

def fba(model: Model, objective: Dict[str, float], lb=None, ub=None, maximize: bool = True) -> Dict:
    lb = model.lb if lb is None else lb
    ub = model.ub if ub is None else ub
    c = np.zeros(model.n)
    for r, coef in objective.items():
        c[model.index(r)] = coef
    res = _solve(-c if maximize else c, None, None, model.S, np.zeros(model.S.shape[0]), _bounds(lb, ub))
    return {"v": res.x, "objective_value": float(c @ res.x)}


def pfba(model: Model, objective: Dict[str, float], lb=None, ub=None, maximize: bool = True,
         fraction: float = 1.0) -> Dict:
    """Parsimonious FBA: optimise c'v, then minimise sum |v| with c'v held at fraction*optimum."""
    lb = model.lb if lb is None else lb
    ub = model.ub if ub is None else ub
    opt = fba(model, objective, lb, ub, maximize)
    n = model.n
    c = np.zeros(n)
    for r, coef in objective.items():
        c[model.index(r)] = coef
    target = fraction * opt["objective_value"]
    # variables: v, t (t >= |v|)
    I = sp.eye(n, format="csr")
    A_ub = [sp.hstack([I, -I]), sp.hstack([-I, -I])]
    b_ub = [np.zeros(n), np.zeros(n)]
    row = sp.csr_matrix(np.concatenate([-c if maximize else c, np.zeros(n)])[None, :])
    A_ub.append(row)
    b_ub.append(np.array([-target if maximize else target]) + 1e-7 * max(1, abs(target)))
    res = _solve(np.concatenate([np.zeros(n), np.ones(n)]), sp.vstack(A_ub).tocsr(), np.concatenate(b_ub),
                 sp.hstack([model.S, sp.csr_matrix(model.S.shape)]).tocsr(), np.zeros(model.S.shape[0]),
                 np.vstack([_bounds(lb, ub), np.tile([0.0, np.inf], (n, 1))]))
    return {"v": res.x[:n], "objective_value": float(c @ res.x[:n]), "fba_optimum": opt["objective_value"]}


# ---------------------------------------------------------------------------
# Loop removal and variability
# ---------------------------------------------------------------------------

def boundary_mask(model: Model) -> np.ndarray:
    return np.array([r.startswith(("EX_", "DM_", "sink_", "SK_")) or r in ("ATPM",) for r in model.rxns])


def cycle_free(model: Model, v: np.ndarray, lb=None, ub=None, tol: float = 0.02,
               fixed: Sequence[int] = ()) -> np.ndarray:
    """CycleFreeFlux (Desouki et al., Bioinformatics 2015): keep boundary fluxes fixed, keep every
    internal flux in its current direction and no larger in magnitude, and minimise total internal
    flux. The result has the same exchanges (hence the same medium demand) and no internal cycles.

    ``fixed`` lists extra reaction indices whose flux must not change (e.g. the query reactions of
    a growth state, so that removing cycles cannot undo the requirements it was built to meet)."""
    lb = model.lb if lb is None else lb
    ub = model.ub if ub is None else ub
    v = np.asarray(v, float)
    bnd = boundary_mask(model)
    fixed = np.asarray(fixed, int)
    if len(fixed):
        bnd = bnd.copy()
        bnd[fixed] = True
    lo = np.where(v >= 0, 0.0, v)
    hi = np.where(v >= 0, v, 0.0)
    lo[bnd] = v[bnd]
    hi[bnd] = v[bnd]
    # small tolerance absorbs the numerical steady-state residual of sampled (float32) vectors
    lo, hi = lo - tol, hi + tol
    if len(fixed):                 # requirement reactions are held exactly (they often sit on a threshold)
        lo[fixed], hi[fixed] = v[fixed], v[fixed]
    lo, hi = np.maximum(lo, lb), np.minimum(hi, ub)
    lo = np.minimum(lo, hi)
    c = np.where(bnd, 0.0, np.sign(v))
    try:
        res = _solve(c, None, None, model.S, np.zeros(model.S.shape[0]), _bounds(lo, hi))
    except LPError:
        if not len(fixed):
            raise
        return cycle_free(model, v, lb, ub, tol)  # fall back to plain loop removal
    return res.x


def fva(model: Model, rxns: Sequence[str], lb=None, ub=None) -> pd.DataFrame:
    lb = model.lb if lb is None else lb
    ub = model.ub if ub is None else ub
    out = []
    for r in rxns:
        lo = fba(model, {r: 1.0}, lb, ub, maximize=False)["objective_value"]
        hi = fba(model, {r: 1.0}, lb, ub, maximize=True)["objective_value"]
        out.append({"reaction": r, "min": lo, "max": hi})
    return pd.DataFrame(out)


def cycle_free_all(model: Model, X: np.ndarray, log=print, every: int = 1000) -> np.ndarray:
    """Apply ``cycle_free`` to every sampled state (about 0.04 s per state for a 2,000-reaction GEM)."""
    import time
    out = np.empty_like(X)
    failed = 0
    t0 = time.time()
    for i in range(len(X)):
        try:
            out[i] = cycle_free(model, X[i].astype(float))
        except LPError:
            out[i] = X[i]
            failed += 1
        if log and every and (i + 1) % every == 0:
            log(f"  loop removal: {i + 1}/{len(X)} states ({time.time() - t0:.0f} s)")
    if log:
        log(f"  loop removal done: {len(X)} states, {failed} kept unchanged (LP failure), {time.time() - t0:.0f} s")
    return out

### Autoencoders (NumPy)

Autoencoders of the feasible flux space, written in plain NumPy.

In [ ]:
F32 = np.float32


# ---------------------------------------------------------------------------
# Preprocessing (signed log + robust scaling), as in the original code
# ---------------------------------------------------------------------------

@dataclass
class Preprocessor:
    """Signed-log transform followed by robust scaling (median / IQR) per reaction.

    The IQR is floored at ``min_scale`` (in log units): after loop removal many reactions are
    almost constant, and dividing by a vanishing IQR would turn tiny numerical noise into huge
    scaled values that dominate the reconstruction loss and the distance computations."""
    center: np.ndarray
    scale: np.ndarray
    log_cap: float = 12.0

    @classmethod
    def fit(cls, X: np.ndarray, min_scale: float = 0.1) -> "Preprocessor":
        L = np.sign(X) * np.log1p(np.abs(X))
        q25, q50, q75 = np.percentile(L, [25, 50, 75], axis=0)
        return cls(center=q50.astype(F32), scale=np.maximum(q75 - q25, min_scale).astype(F32))

    def transform(self, X: np.ndarray) -> np.ndarray:
        return ((np.sign(X) * np.log1p(np.abs(X)) - self.center) / self.scale).astype(F32)

    def inverse(self, Y: np.ndarray, return_jac: bool = False):
        u = Y * self.scale + self.center
        clipped = np.abs(u) > self.log_cap
        u = np.clip(u, -self.log_cap, self.log_cap)
        V = np.sign(u) * np.expm1(np.abs(u))
        if not return_jac:
            return V
        dV_dY = np.where(clipped, 0.0, np.exp(np.abs(u)) * self.scale).astype(F32)
        return V, dV_dY


# ---------------------------------------------------------------------------
# Dense network with manual back-propagation
# ---------------------------------------------------------------------------

class Dense:
    """Stack of affine layers with leaky-ReLU between them (linear output)."""

    def __init__(self, sizes: List[int], rng: np.random.Generator, slope: float = 0.01):
        self.slope = slope
        self.W, self.b = [], []
        for a, c in zip(sizes[:-1], sizes[1:]):
            self.W.append((rng.standard_normal((a, c)) * np.sqrt(2.0 / a)).astype(F32))
            self.b.append(np.zeros(c, F32))

    def params(self):
        return self.W + self.b

    def forward(self, x, cache: bool = False):
        hs, h = [x], x
        for k, (W, b) in enumerate(zip(self.W, self.b)):
            z = h @ W + b
            h = z if k == len(self.W) - 1 else np.where(z > 0, z, self.slope * z)
            hs.append(z)
            if k < len(self.W) - 1:
                hs.append(h)
        return (h, hs) if cache else h

    def backward(self, hs, g_out):
        """Return (grads matching params(), gradient w.r.t. the input)."""
        gW = [None] * len(self.W)
        gb = [None] * len(self.b)
        g = g_out
        n_layers = len(self.W)
        # hs = [x, z0, h0, z1, h1, ..., z_last]
        for k in reversed(range(n_layers)):
            h_in = hs[0] if k == 0 else hs[2 * k]
            gW[k] = h_in.T @ g
            gb[k] = g.sum(axis=0)
            g = g @ self.W[k].T
            if k > 0:
                z_prev = hs[2 * k - 1]
                g = g * np.where(z_prev > 0, 1.0, self.slope).astype(F32)
        return gW + gb, g


class Autoencoder:
    """Dense autoencoder of scaled flux vectors (encoder n -> 512 -> 128 -> latent, mirrored decoder).
    The decoder emits scaled fluxes; ``decode_flux`` maps them back to flux units."""

    def __init__(self, n_in: int, pre: Preprocessor, latent: int = 256, hidden=(512, 128), seed: int = 42):
        rng = np.random.default_rng(seed)
        self.pre = pre
        self.enc = Dense([n_in, *hidden, latent], rng)
        self.dec = Dense([latent, *hidden[::-1], n_in], rng)
        self.latent = latent

    def params(self):
        return self.enc.params() + self.dec.params()

    def encode(self, X):
        return self.enc.forward(X)

    def decode_all(self, Z, cache: bool = False):
        """Return (scaled fluxes Y, fluxes V, cache for back-propagation)."""
        Y, hs = self.dec.forward(Z, cache=True)
        V, J = self.pre.inverse(Y, return_jac=True)
        return Y, V, (hs, J)

    def decode(self, Z):
        return self.decode_all(Z)[0]

    def decode_flux(self, Z):
        return self.decode_all(Z)[1]

    def backward_decoder(self, cache, gY=None, gV=None):
        """Gradients of the decoder parameters and of Z given dL/dY and/or dL/dV."""
        hs, J = cache
        g = (0 if gY is None else gY) + (0 if gV is None else gV * J)
        return self.dec.backward(hs, np.asarray(g, F32))

    def decode_backward(self, Z, gV):
        _, _, cache = self.decode_all(Z)
        return self.backward_decoder(cache, gV=gV)[1]

    def state(self):
        return [p.copy() for p in self.params()]

    def load(self, state):
        for p, s in zip(self.params(), state):
            p[...] = s


# ---------------------------------------------------------------------------
# Physics-informed penalties (mass balance and model bounds) in flux space
# ---------------------------------------------------------------------------

@dataclass
class Physics:
    S: sp.csr_matrix
    lb: np.ndarray
    ub: np.ndarray
    flux_scale: float
    rxn_scale: np.ndarray

    @classmethod
    def from_model(cls, model, X: np.ndarray) -> "Physics":
        q25, q75 = np.percentile(X, [25, 75], axis=0)
        return cls(S=model.S.astype(F32).tocsr(), lb=model.lb.astype(F32), ub=model.ub.astype(F32),
                   flux_scale=float(max(np.median(np.abs(X)), 1.0)),
                   rxn_scale=np.maximum(q75 - q25, 1.0).astype(F32))

    def penalties(self, V: np.ndarray):
        """Return (mass_loss, bounds_loss, d_mass/dV, d_bounds/dV) averaged over the batch."""
        B = V.shape[0]
        R = (self.S @ V.T).T / self.flux_scale                     # B x m
        mass = float(np.mean(R ** 2))
        g_mass = (2.0 / R.size) * (self.S.T @ R.T).T / self.flux_scale
        low = np.maximum(self.lb - V, 0.0)
        up = np.maximum(V - self.ub, 0.0)
        E = (low + up) / self.rxn_scale
        bounds = float(np.mean(E ** 2))
        g_b = (2.0 / E.size) * E / self.rxn_scale * (np.where(up > 0, 1.0, 0.0) - np.where(low > 0, 1.0, 0.0))
        return mass, bounds, g_mass.astype(F32), g_b.astype(F32)


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

@dataclass
class TrainConfig:
    epochs: int = 80
    batch_size: int = 256
    lr: float = 3e-4
    onecycle_factor: float = 2.0
    weight_decay: float = 1e-4
    val_split: float = 0.3
    alpha_mse: float = 0.8
    beta_huber: float = 0.2
    latent_l1: float = 1e-4
    lambda_mass: float = 0.0      # > 0 turns the AE into the PI-AE
    lambda_bounds: float = 0.0
    physics_warmup: float = 0.3   # fraction of epochs over which the physics weights ramp up linearly
    clip_norm: float = 1.0
    seed: int = 42
    verbose_every: int = 10


def _onecycle(step, total, max_lr, pct=0.3, div=25.0, final_div=1e4):
    warm = max(1, int(pct * total))
    if step < warm:
        return max_lr / div + (max_lr - max_lr / div) * step / warm
    t = (step - warm) / max(1, total - warm)
    return max_lr / final_div + (max_lr - max_lr / final_div) * 0.5 * (1 + np.cos(np.pi * t))


def _rec_loss(Y, X, Z, cfg):
    d = Y - X
    ad = np.abs(d)
    mse = np.mean(d ** 2)
    hub = np.mean(np.where(ad <= 1.0, 0.5 * d ** 2, ad - 0.5))
    l1 = np.mean(np.abs(Z))
    loss = cfg.alpha_mse * mse + cfg.beta_huber * hub + cfg.latent_l1 * l1
    gY = (cfg.alpha_mse * 2 * d + cfg.beta_huber * np.clip(d, -1, 1)) / d.size
    gZ = cfg.latent_l1 * np.sign(Z) / Z.size
    return loss, mse, gY.astype(F32), gZ.astype(F32)


def train(model: Autoencoder, Xs: np.ndarray, cfg: TrainConfig, physics: Optional[Physics] = None,
          log: Callable = print) -> Dict:
    """Train an AE (or PI-AE when cfg.lambda_mass/lambda_bounds > 0) on scaled fluxes Xs.
    The weights of the epoch with the lowest validation loss are kept."""
    Xt = Xs
    rng = np.random.default_rng(cfg.seed)
    idx = rng.permutation(len(Xs))
    n_val = max(1, int(cfg.val_split * len(Xs)))
    va, tr = idx[:n_val], idx[n_val:]
    params = model.params()
    m = [np.zeros_like(p) for p in params]
    v = [np.zeros_like(p) for p in params]
    steps_per_epoch = int(np.ceil(len(tr) / cfg.batch_size))
    total = cfg.epochs * steps_per_epoch
    use_phys = physics is not None and (cfg.lambda_mass > 0 or cfg.lambda_bounds > 0)
    hist = {k: [] for k in ["train_loss", "val_loss", "val_rec_mse", "val_mass", "val_bounds"]}
    best, best_state, best_epoch, step = np.inf, model.state(), 0, 0
    t0 = time.time()

    def evaluate(ids):
        tot = rec = mass = bnd = 0.0
        for s in range(0, len(ids), 2048):
            xb = Xs[ids[s:s + 2048]]
            z = model.encode(xb)
            y, Vb, _ = model.decode_all(z)
            l, mse, _, _ = _rec_loss(y, Xt[ids[s:s + 2048]], z, cfg)
            w = len(xb)
            rec += mse * w
            tot += l * w
            if use_phys:
                ms, bd, _, _ = physics.penalties(Vb)
                mass += ms * w
                bnd += bd * w
                tot += (cfg.lambda_mass * ms + cfg.lambda_bounds * bd) * w
        n = len(ids)
        return tot / n, rec / n, mass / n, bnd / n

    for epoch in range(1, cfg.epochs + 1):
        # physics penalties are switched on gradually: early decoders are far from steady state and a
        # full-strength penalty at that point destabilises training (loss spikes)
        ramp = min(1.0, epoch / max(1.0, cfg.physics_warmup * cfg.epochs))
        rng.shuffle(tr)
        run, nb = 0.0, 0
        for s in range(0, len(tr), cfg.batch_size):
            bi = tr[s:s + cfg.batch_size]
            xb = Xs[bi]
            z, hz = model.enc.forward(xb, cache=True)
            y, Vb, cache = model.decode_all(z, cache=True)
            loss, _, gY, gZ_l1 = _rec_loss(y, Xt[bi], z, cfg)
            gV = None
            if use_phys:
                ms, bd, gm, gb = physics.penalties(Vb)
                lm, lbd = ramp * cfg.lambda_mass, ramp * cfg.lambda_bounds
                loss += lm * ms + lbd * bd
                gV = lm * gm + lbd * gb
            g_dec, gZ = model.backward_decoder(cache, gY=gY, gV=gV)
            g_enc, _ = model.enc.backward(hz, gZ + gZ_l1)
            grads = g_enc + g_dec
            gn = np.sqrt(sum(float(np.sum(g * g)) for g in grads))
            if gn > cfg.clip_norm:
                grads = [g * (cfg.clip_norm / gn) for g in grads]
            step += 1
            lr = _onecycle(step - 1, total, cfg.lr * cfg.onecycle_factor)
            for p, g, mi, vi in zip(params, grads, m, v):
                mi *= 0.9; mi += 0.1 * g
                vi *= 0.999; vi += 0.001 * g * g
                mhat = mi / (1 - 0.9 ** step)
                vhat = vi / (1 - 0.999 ** step)
                p -= (lr * (mhat / (np.sqrt(vhat) + 1e-8) + cfg.weight_decay * p)).astype(F32)
            run += float(loss)
            nb += 1
        vt, vr, vm, vb = evaluate(va)
        hist["train_loss"].append(run / nb)
        hist["val_loss"].append(vt)
        hist["val_rec_mse"].append(vr)
        hist["val_mass"].append(vm)
        hist["val_bounds"].append(vb)
        if vt < best:
            best, best_state, best_epoch = vt, model.state(), epoch
        if cfg.verbose_every and (epoch % cfg.verbose_every == 0 or epoch == 1):
            log(f"  epoch {epoch:3d}/{cfg.epochs}  train={run / nb:.4f}  val={vt:.4f}  "
                f"val_rec={vr:.4f}" + (f"  mass={vm:.3g}  bounds={vb:.3g}" if use_phys else "")
                + f"  ({time.time() - t0:.0f}s)")
    model.load(best_state)
    hist.update(best_epoch=best_epoch, best_val_loss=best, val_index=va, train_index=tr,
                seconds=time.time() - t0)
    return hist


def reconstruct(model: Autoencoder, Xs: np.ndarray, batch: int = 4096) -> np.ndarray:
    out = np.empty_like(Xs)
    for s in range(0, len(Xs), batch):
        out[s:s + batch] = model.decode(model.encode(Xs[s:s + batch]))
    return out

### FluxQuery latent search

FluxQuery: phenotype-directed search in the latent space of a trained autoencoder.

In [ ]:
def choose_seeds(X: np.ndarray, cons: pd.DataFrame, n_seeds: int = 32, top_fraction: float = 0.05,
                 seed: int = 0) -> np.ndarray:
    """Pick seeds at random among the top `top_fraction` sampled states by soft score
    (diverse starting points that are already partly growth-like)."""
    _, viol, _ = compliance(X, cons)
    order = np.argsort(viol)
    pool = order[:max(n_seeds, int(top_fraction * len(X)))]
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(pool, size=min(n_seeds, len(pool)), replace=False))


def latent_search(model: Autoencoder, physics: Physics, cons: pd.DataFrame, X_seed_scaled: np.ndarray,
        steps: int = 400, lr: float = 2e-2, lambda_bio: float = 10.0, lambda_latent: float = 0.05,
        lambda_mass="auto", lambda_bounds="auto", physics_share: float = 0.1,
        log_every: int = 100, log=print) -> Dict:
    """Optimise the latent codes of the seeds. With lambda_mass/lambda_bounds = "auto" the physics
    terms are rescaled so that, at the start, each contributes `physics_share` of the biological
    term: decoded fluxes carry large steady-state residuals (reconstruction error is amplified by
    the inverse log transform), and un-normalised mass penalties otherwise swamp the biological
    signal. Exact feasibility is restored afterwards by linear programming."""
    Z0 = model.encode(X_seed_scaled.astype(F32))
    V0 = model.decode_flux(Z0)
    b0, _ = bio_penalty_and_grad(V0, cons, X_seed_scaled.shape[1])
    m0, bd0, _, _ = physics.penalties(V0)
    if lambda_mass == "auto":
        lambda_mass = physics_share * lambda_bio * b0 / max(m0, 1e-12)
    if lambda_bounds == "auto":
        lambda_bounds = physics_share * lambda_bio * b0 / max(bd0, 1e-12)
    Z = Z0.copy()
    m = np.zeros_like(Z)
    v = np.zeros_like(Z)
    n = X_seed_scaled.shape[1]
    hist = {k: [] for k in ["total", "bio", "latent", "mass", "bounds"]}
    for t in range(1, steps + 1):
        V = model.decode_flux(Z)
        lb_, gb_ = bio_penalty_and_grad(V, cons, n)
        ms, bd, gm, gbd = physics.penalties(V)
        ll = float(np.mean((Z - Z0) ** 2))
        gV = lambda_bio * gb_ + lambda_mass * gm + lambda_bounds * gbd
        gZ = model.decode_backward(Z, gV.astype(F32)) + lambda_latent * 2 * (Z - Z0) / Z.size
        m = 0.9 * m + 0.1 * gZ
        v = 0.999 * v + 0.001 * gZ * gZ
        Z -= (lr * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)).astype(F32)
        tot = lambda_bio * lb_ + lambda_latent * ll + lambda_mass * ms + lambda_bounds * bd
        for k, val in zip(hist, [tot, lb_, ll, ms, bd]):
            hist[k].append(val)
        if log_every and (t % log_every == 0 or t == 1):
            log(f"  step {t:4d}/{steps}  total={tot:.4g}  bio={lb_:.4g}  latent={ll:.3g}  mass={ms:.3g}  bounds={bd:.3g}")
    V_before = model.decode_flux(Z0)
    V_after = model.decode_flux(Z)
    return {"V_before": V_before, "V_after": V_after, "Z0": Z0, "Z": Z, "history": pd.DataFrame(hist),
            "lambdas": {"bio": lambda_bio, "latent": lambda_latent, "mass": float(lambda_mass),
                        "bounds": float(lambda_bounds)}}

### Evaluation

Evaluation helpers: reconstruction quality, manifold proximity and candidate summaries.

In [ ]:
def reconstruction_report(model: Autoencoder, Xs_val: np.ndarray, X_val: np.ndarray,
                          gem: Model) -> Dict[str, float]:
    R = reconstruct(model, Xs_val)
    T = Xs_val
    ss = ((T - R) ** 2).sum(0)
    st = ((T - T.mean(0)) ** 2).sum(0)
    ok = st > 1e-8
    V = np.vstack([model.decode_flux(model.encode(Xs_val[s:s + 4096])) for s in range(0, len(Xs_val), 4096)])
    res = np.abs(gem.S @ V.T.astype(float))
    res_true = np.abs(gem.S @ X_val.T.astype(float))
    return {
        "R2_global_target_space": float(1 - ss[ok].sum() / st[ok].sum()),
        "R2_median_reaction": float(np.median(1 - ss[ok] / st[ok])),
        "MAE_flux": float(np.mean(np.abs(V - X_val))),
        "median_abs_Sv_decoded": float(np.median(res)),
        "median_abs_Sv_sampled": float(np.median(res_true)),
    }


class Proximity:
    """Distance of a flux vector to the sampled flux space, in the scaled coordinates the
    networks see, expressed as a percentile of the sample-to-sample nearest-neighbour distances."""

    def __init__(self, pre: Preprocessor, X: np.ndarray, n_ref: int = 20000, seed: int = 0):
        rng = np.random.default_rng(seed)
        ref = X if len(X) <= n_ref else X[rng.choice(len(X), n_ref, replace=False)]
        self.pre = pre
        self.ref = pre.transform(ref)
        self.nn = NearestNeighbors(n_neighbors=2).fit(self.ref)
        d, _ = self.nn.kneighbors(self.ref)
        self.null = np.sort(d[:, 1])           # leave-one-out NN distances within the sampling

    def __call__(self, V: np.ndarray) -> pd.DataFrame:
        d, _ = self.nn.kneighbors(self.pre.transform(np.atleast_2d(V)), n_neighbors=1)
        d = d[:, 0]
        pct = np.searchsorted(self.null, d) / len(self.null) * 100
        return pd.DataFrame({"nn_distance": d, "nn_distance_percentile": pct})


def summarise(name: str, V: np.ndarray, cons: pd.DataFrame, gem: Model, prox: Optional[Proximity] = None,
              start: Optional[np.ndarray] = None, scale: Optional[np.ndarray] = None) -> pd.DataFrame:
    """One row per flux vector: compliance, soft score, steady-state residual, bound violation,
    distance to the sampled space and (optionally) normalised L1 change from a starting vector."""
    V = np.atleast_2d(V)
    frac, mv, soft = compliance(V, cons)
    res = np.abs(gem.S @ V.T).max(axis=0)
    bviol = np.maximum(np.maximum(gem.lb - V, 0), np.maximum(V - gem.ub, 0)).max(axis=1)
    df = pd.DataFrame({"method": name, "compliance": frac, "soft_score": soft, "max_abs_Sv": res,
                       "max_bound_violation": bviol})
    if prox is not None:
        df = pd.concat([df, prox(V).reset_index(drop=True)], axis=1)
    if start is not None:
        sc = np.ones(V.shape[1]) if scale is None else scale
        df["normalised_L1_change"] = np.sum(np.abs(V - np.atleast_2d(start)) / sc, axis=1)
    return df

### Classical objective functions

Classical single-objective predictions (FBA / parsimonious FBA) for comparison with FluxQuery.

In [ ]:
def default_objectives(model: Model, context: str) -> Dict[str, Dict[str, float]]:
    """Objectives commonly used for neurons, restricted to reactions present in the model."""
    identity = {"dopaminergic": "DOPAVESSEC", "glutamatergic": "GLUVESSEC", "cholinergic": "ACHVESSEC"}.get(context)
    atp = "ATPM" if model.has("ATPM") else "DM_atp_c_"
    membrane = [r for r in ("CEPTC", "PSDm_hs", "PSSA2_hs", "SMS", "HMR_1467") if model.has(r)]
    obj = {
        "max ATP demand (pFBA)": {atp: 1.0},
        "max membrane lipid synthesis (pFBA)": {r: 1.0 for r in membrane},
        "max NADPH via PPP (pFBA)": {"G6PDH2c": 1.0} if model.has("G6PDH2c") else {},
        "min total flux at ATP lower bound (pFBA)": {atp: -1.0},
    }
    if identity and model.has(identity):
        obj[f"max neurotransmitter release: {identity} (pFBA)"] = {identity: 1.0}
    return {k: v for k, v in obj.items() if v}


def run_objectives(model: Model, objectives: Dict[str, Dict[str, float]], lb=None, ub=None):
    V, rows = [], []
    for name, obj in objectives.items():
        try:
            r = pfba(model, obj, lb, ub, maximize=True)
            V.append(r["v"])
            rows.append({"method": name, "objective_value": r["objective_value"]})
        except LPError as exc:
            rows.append({"method": name, "objective_value": np.nan, "error": str(exc)})
    return (np.array(V) if V else np.zeros((0, model.n))), pd.DataFrame(rows)

### Medium design

From growth flux states to a culture medium.

In [ ]:
GASES_AND_SOLVENT = ("EX_h2o[e]", "EX_h[e]", "EX_o2[e]", "EX_co2[e]")


# ---------------------------------------------------------------------------
# Medium description
# ---------------------------------------------------------------------------

@dataclass
class Medium:
    table: pd.DataFrame          # columns: compound, rxn, group, base_uM, max_uptake
    k: float                     # (umol/gDW/h) per (umol/L)

    @classmethod
    def from_bibliomic(cls, path) -> "Medium":
        """Read mediaData + cellCultureData from a bibliomic workbook."""
        md = pd.read_excel(path, sheet_name="mediaData")
        cc = pd.read_excel(path, sheet_name="cellCultureData").iloc[0]
        gdw = cc["averageProteinConcentration"] * cc["assayVolume"] / cc["proteinFraction"]
        k = cc["volume"] / (cc["interval"] * gdw)
        t = pd.DataFrame({"compound": md["Compound"].astype(str).str.strip(),
                          "rxn": md["rxns"].astype(str).str.strip(),
                          "group": md["Group"].astype(str).str.strip(),
                          "base_uM": pd.to_numeric(md["mediumConcentrations"], errors="coerce")})
        t["max_uptake"] = k * t["base_uM"]
        check = np.nanmedian(np.abs(md["mediumMaxUptake"]) / t["max_uptake"])
        if not 0.98 < check < 1.02:
            raise ValueError(f"Culture parameters do not reproduce mediumMaxUptake (ratio {check:.3f}).")
        return cls(table=t.reset_index(drop=True), k=float(k))

    def uM(self, uptake: np.ndarray) -> np.ndarray:
        return np.asarray(uptake) / self.k


def consistent_bounds(model: Model, medium: Medium, keep: Sequence[str] = GASES_AND_SOLVENT,
                      slack: float = 1.05):
    """Close artefactual uptakes of non-medium metabolites; return (lb, ub, report)."""
    lb, ub = model.lb.copy(), model.ub.copy()
    in_medium = set(medium.table["rxn"])
    closed = []
    for i, r in enumerate(model.rxns):
        if r.startswith("EX_") and lb[i] < 0 and r not in in_medium and r not in keep and ub[i] >= 0:
            closed.append(i)
    lb[closed] = 0.0
    try:
        _solve(np.zeros(model.n), None, None, model.S, np.zeros(model.S.shape[0]), np.column_stack([lb, ub]))
        needed = {}
    except LPError:
        # minimal re-opening of the closed uptakes that the network needs (min total uptake)
        k = len(closed)
        A = sp.lil_matrix((k, model.n + k))
        for t, i in enumerate(closed):
            A[t, i] = -1.0
            A[t, model.n + t] = -1.0
        lb2 = model.lb.copy()
        res = _solve(np.concatenate([np.zeros(model.n), np.ones(k)]), A.tocsr(), np.zeros(k),
                     sp.hstack([model.S, sp.csr_matrix((model.S.shape[0], k))]).tocsr(), np.zeros(model.S.shape[0]),
                     np.vstack([np.column_stack([lb2, ub]), np.tile([0, np.inf], (k, 1))]))
        u = res.x[model.n:]
        needed = {model.rxns[i]: float(u[t]) for t, i in enumerate(closed) if u[t] > 1e-9}
        for r, val in needed.items():
            lb[model.index(r)] = -val * slack
    rep = pd.DataFrame({"exchange": [model.rxns[i] for i in closed],
                        "original_lb": [model.lb[i] for i in closed],
                        "new_lb": [lb[i] for i in closed]})
    rep["status"] = np.where(rep["new_lb"] < 0, "minimal uptake kept (required for feasibility)", "uptake closed")
    return lb, ub, rep


def classify_components(model: Model, medium: Medium, lb: np.ndarray, tol: float = 0.95) -> pd.DataFrame:
    """Is each medium component limited by its availability in the medium (model lower bound ~
    -max_uptake) or by a measured/transport rate (model bound tighter than availability)?"""
    t = medium.table.copy()
    t["in_model"] = t["rxn"].map(model.has)
    t["model_lb"] = [lb[model.index(r)] if ok else np.nan for r, ok in zip(t["rxn"], t["in_model"])]
    t["model_ub"] = [model.ub[model.index(r)] if ok else np.nan for r, ok in zip(t["rxn"], t["in_model"])]
    ratio = -t["model_lb"] / t["max_uptake"]
    t["limitation"] = np.select(
        [~t["in_model"], t["model_lb"] >= 0, ratio >= tol],
        ["not in model", "uptake not allowed by model", "availability-limited"],
        "rate-limited (measured uptake)")
    return t


# ---------------------------------------------------------------------------
# Demand of growth states versus the background flux space
# ---------------------------------------------------------------------------

def demand_table(model: Model, medium: Medium, V_growth: np.ndarray, V_background: np.ndarray,
                 lb: np.ndarray) -> pd.DataFrame:
    t = classify_components(model, medium, lb)
    rows = []
    for _, r in t.iterrows():
        if not r["in_model"]:
            rows.append({})
            continue
        j = model.index(r["rxn"])
        g, b = -V_growth[:, j], -V_background[:, j]          # uptake = -v
        pooled = np.concatenate([g, b])
        # Cliff's delta (rank-based effect size, robust to non-normal flux distributions)
        rk = pd.Series(pooled).rank().to_numpy()
        auc = (rk[:len(g)].sum() - len(g) * (len(g) + 1) / 2) / (len(g) * len(b))
        rows.append({"uptake_growth_median": np.median(g), "uptake_growth_p90": np.percentile(g, 90),
                     "uptake_background_median": np.median(b), "cliffs_delta": 2 * auc - 1,
                     "fraction_of_availability_growth": np.median(g) / r["max_uptake"] if r["max_uptake"] > 0 else np.nan})
    return pd.concat([t, pd.DataFrame(rows)], axis=1)


# ---------------------------------------------------------------------------
# Medium design by linear programming
# ---------------------------------------------------------------------------

def design_lp(model: Model, cons: pd.DataFrame, medium: Medium, lb: np.ndarray, ub: np.ndarray,
              enrichable: Sequence[str], budget: float, max_fold: float = 10.0,
              median_sign: Optional[np.ndarray] = None, supplements: Optional[Dict[str, float]] = None,
              gamma: float = 1e-4) -> Dict:
    """Minimise the biological violation of the growth query when the concentrations of the
    `enrichable` medium components may be raised by fold-increments e_i (0 <= e_i <= max_fold - 1)
    with sum_i e_i <= budget. A tiny cost gamma * sum e_i picks the leanest of equally good media.

    Enriching component i by a factor (1 + e_i) scales its *current* uptake capacity (-lb_i):
    for availability-limited components this is exactly the extra amount supplied by the
    medium; for rate-limited components it assumes the transporter is not saturated (a
    hypothesis, used only in sensitivity analyses).

    `supplements` maps exchange IDs of compounds absent from the base medium to a test
    concentration (umol/L); their uptake is allowed up to k * concentration.
    """
    n = model.n
    lb, ub = lb.copy(), ub.copy()
    if supplements:
        for r, c_uM in supplements.items():
            if model.has(r):
                lb[model.index(r)] = min(lb[model.index(r)], -medium.k * c_uM)
    act, sign, w = _bio_rows(cons, n, median_sign)
    kk = len(act)
    j = act["rxn_index"].to_numpy(int)
    tl, tu = act["target_lb"].to_numpy(float), act["target_ub"].to_numpy(float)
    tab = medium.table.set_index("rxn")
    enr = [r for r in enrichable if model.has(r)]
    ne = len(enr)
    nv = n + 2 * kk + ne
    c = np.zeros(nv)
    c[n:n + kk] = w
    c[n + kk:n + 2 * kk] = w
    c[n + 2 * kk:] = gamma
    rows, rhs = [], []
    L = sp.lil_matrix((kk, nv))
    U = sp.lil_matrix((kk, nv))
    for r in range(kk):
        L[r, j[r]] = -sign[r]; L[r, n + r] = -1.0
        U[r, j[r]] = sign[r]; U[r, n + kk + r] = -1.0
    hl, hu = np.isfinite(tl), np.isfinite(tu)
    rows += [L.tocsr()[hl], U.tocsr()[hu]]
    rhs += [-tl[hl], tu[hu]]
    # enrichment: v_i >= -base_uptake_i * (1 + e_i)   <=>   -v_i - base_i e_i <= base_i
    E = sp.lil_matrix((ne + 1, nv))
    b_e = np.zeros(ne + 1)
    lb_e = lb.copy()
    for t, r in enumerate(enr):
        i = model.index(r)
        base = max(-lb[i], 0.0)
        E[t, i] = -1.0
        E[t, n + 2 * kk + t] = -base
        b_e[t] = base
        lb_e[i] = -base * max_fold
    E[ne, n + 2 * kk:] = 1.0
    b_e[ne] = budget
    rows.append(E.tocsr())
    rhs.append(b_e)
    bounds = np.vstack([np.column_stack([lb_e, ub]), np.tile([0, np.inf], (2 * kk, 1)),
                        np.tile([0, max_fold - 1], (ne, 1))])
    A_eq = sp.hstack([model.S, sp.csr_matrix((model.S.shape[0], nv - n))]).tocsr()
    res = _solve(c, sp.vstack(rows).tocsr(), np.concatenate(rhs), A_eq, np.zeros(model.S.shape[0]), bounds)
    x = res.x
    v = x[:n]
    viol = float(np.sum(w * (x[n:n + kk] + x[n + kk:n + 2 * kk])))
    fold = pd.Series(1 + x[n + 2 * kk:], index=enr)
    return {"v": v, "violation": viol, "fold": fold, "compliance": float(compliance(v, cons)[0][0])}


def budget_scan(model, cons, medium, lb, ub, enrichable, budgets=(0, 0.5, 1, 2, 4, 8, 16), **kw) -> pd.DataFrame:
    out = []
    for B in budgets:
        r = design_lp(model, cons, medium, lb, ub, enrichable, B, **kw)
        row = {"budget": B, "violation": r["violation"], "compliance": r["compliance"]}
        row.update({f"fold:{k}": v for k, v in r["fold"].items()})
        out.append(row)
    return pd.DataFrame(out)


def choose_budget(scan: pd.DataFrame, slope_fraction: float = 0.25) -> float:
    """Knee of the budget scan: the largest budget reached while every additional unit of
    enrichment still buys at least `slope_fraction` of the improvement the first unit bought.
    Beyond the knee, extra concentration gives diminishing returns."""
    b = scan["budget"].to_numpy(float)
    v = scan["violation"].to_numpy(float)
    slopes = (v[:-1] - v[1:]) / np.maximum(b[1:] - b[:-1], 1e-12)
    if len(slopes) == 0 or slopes[0] <= 1e-12:
        return 0.0
    chosen = b[0]
    for k, sl in enumerate(slopes):
        if sl < slope_fraction * slopes[0]:
            break
        chosen = b[k + 1]
    return float(chosen)


def supplement_screen(model, cons, medium, lb, ub, candidates: Dict[str, str], dose_uM: float = 100.0,
                      enrichable=(), budget: float = 0.0, **kw) -> pd.DataFrame:
    """Effect of adding one compound at a time (test dose) on the growth-query violation."""
    ref = design_lp(model, cons, medium, lb, ub, enrichable, budget, **kw)["violation"]
    out = []
    for rxn, name in candidates.items():
        if not model.has(rxn):
            out.append({"exchange": rxn, "compound": name, "status": "no exchange in model"})
            continue
        i = model.index(rxn)
        if model.lb[i] > 0:
            out.append({"exchange": rxn, "compound": name,
                        "status": "net secretion measured in this cell type; uptake not modelled"})
            continue
        r = design_lp(model, cons, medium, lb, ub, enrichable, budget, supplements={rxn: dose_uM}, **kw)
        out.append({"exchange": rxn, "compound": name, "status": "tested", "violation": r["violation"],
                    "violation_reduction_%": 100 * (ref - r["violation"]) / ref if ref > 0 else 0.0,
                    "uptake_at_optimum": -r["v"][i]})
    return pd.DataFrame(out)


#: Known model artefacts that can make the LP enrich a component for the wrong reason.
#: Keys are exchange reactions; values are the note shown next to the recommendation.
BIOLOGICAL_CAVEATS = {
    "EX_phe_L[e]": "acts in the model through phenylalanine hydroxylase (PAH), a hepatic enzyme with negligible "
                   "neuronal expression; the biological lever is tyrosine (supply for TH) - test tyrosine instead",
    "EX_gal[e]": "galactose is a phantom carbon source in the uniform sampling of this model",
    "EX_34dhphe[e]": "L-DOPA is a drug, not a nutrient; uptake in the model bypasses tyrosine hydroxylase",
}


def formulation(medium: Medium, classes: pd.DataFrame, fold: pd.Series, demand: Optional[pd.DataFrame] = None,
                supplements: Optional[pd.DataFrame] = None, min_gain: float = 0.05,
                caveats: Optional[Dict[str, str]] = None,
                fold_transport: Optional[pd.Series] = None) -> pd.DataFrame:
    """Final recommended medium: base concentrations, LP-derived enrichments, and supplements.
    Components the LP enriches through a known model artefact are flagged in the `note` column.
    `fold_transport` (optional) is the design obtained when rate-limited components may also scale
    (hypothesis: transport not saturated); it is reported as a second tier, `uM_if_transport_unsaturated`."""
    caveats = BIOLOGICAL_CAVEATS if caveats is None else caveats
    t = classes[["compound", "group", "rxn", "base_uM", "limitation"]].copy()
    t["fold_change"] = t["rxn"].map(fold).fillna(1.0)
    t.loc[t["fold_change"] < 1 + min_gain, "fold_change"] = 1.0
    t["recommended_uM"] = t["base_uM"] * t["fold_change"]
    t["recommendation"] = np.where(t["fold_change"] > 1, "increase",
                                   np.where(t["limitation"].str.startswith("availability"),
                                            "keep (enrichment does not improve growth)", "keep"))
    t["note"] = t["rxn"].map(caveats).fillna("")
    if fold_transport is not None:
        ft = t["rxn"].map(fold_transport).fillna(1.0)
        ft[ft < 1 + min_gain] = 1.0
        t["fold_if_transport_unsaturated"] = ft
        t["uM_if_transport_unsaturated"] = t["base_uM"] * ft
    flagged = (t["fold_change"] > 1) & (t["note"] != "")
    t.loc[flagged, "recommendation"] = "increase in the model; see note"
    if demand is not None:
        t = t.merge(demand[["rxn", "uptake_growth_median", "uptake_background_median", "cliffs_delta"]],
                    on="rxn", how="left")
    if supplements is not None and len(supplements):
        add = supplements[(supplements["status"] == "tested") & (supplements["violation_reduction_%"] > 1)]
        extra = pd.DataFrame({"compound": add["compound"], "group": "supplement", "rxn": add["exchange"],
                              "base_uM": 0.0, "limitation": "absent from base medium", "fold_change": np.nan,
                              "recommended_uM": np.nan, "recommendation": "add (see supplement screen)",
                              "note": add["exchange"].map(caveats).fillna("")})
        t = pd.concat([t, extra], ignore_index=True)
    return t

### Figures

Figures for the notebook (matplotlib; consistent, colour-blind-validated palette).

In [ ]:
# Categorical slots in fixed order (validated palette; see dataviz reference palette).
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GREY, INK, INK2 = "#9b9a96", "#0b0b0b", "#52514e"
SEQ = mpl.colors.LinearSegmentedColormap.from_list("blue", ["#f2f6fc", "#9cc0ec", "#2a78d6", "#123a6b"])
DIV = mpl.colors.LinearSegmentedColormap.from_list("div", ["#c2410c", "#f4f3ef", "#2a78d6"])


def style():
    mpl.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
        "font.size": 9.5, "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.labelsize": 9.5,
        "axes.edgecolor": "#c9c8c3", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
        "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#ecebe7",
        "grid.linewidth": 0.6, "legend.frameon": False, "axes.prop_cycle": mpl.cycler(color=C),
        "figure.facecolor": "white", "axes.facecolor": "white", "lines.linewidth": 2,
    })


def _save(fig, path):
    if path is not None:
        fig.savefig(path)
    return fig


def module_colors(modules: Sequence[str]) -> Dict[str, str]:
    return {m: C[i % len(C)] for i, m in enumerate(dict.fromkeys(modules))}


# ---------------------------------------------------------------------------
def query_thresholds(X: np.ndarray, cons: pd.DataFrame, path=None):
    """Activity distribution of every requirement in the sampled space (P5-P95 bar, IQR box,
    median tick) with the query window shaded. Activities are standardised by their IQR so
    reactions with very different flux scales share one axis."""
    cons = cons.reset_index(drop=True)
    A = activity(X[:, cons["rxn_index"].to_numpy(int)], cons["direction"].to_numpy())
    med = np.median(A, axis=0)
    s = cons["scale"].to_numpy(float)
    Z = (A - med) / s
    cols = module_colors(cons["module"])
    fig, ax = plt.subplots(figsize=(7.2, 0.19 * len(cons) + 1.2))
    for k, r in cons.iterrows():
        y = len(cons) - 1 - k
        p5, p25, p75, p95 = np.percentile(Z[:, k], [5, 25, 75, 95])
        col = cols[r["module"]] if r["status"] == "active" else GREY
        lo = (r["target_lb"] - med[k]) / s[k] if np.isfinite(r["target_lb"]) else p5
        hi = (r["target_ub"] - med[k]) / s[k] if np.isfinite(r["target_ub"]) else max(p95, lo) + 0.4
        ax.axhspan(y - 0.42, y + 0.42, xmin=0, xmax=1, color="none")
        ax.fill_betweenx([y - 0.38, y + 0.38], lo, hi, color=col, alpha=0.13, lw=0)
        ax.plot([p5, p95], [y, y], color=col, lw=1.2)
        ax.plot([p25, p75], [y, y], color=col, lw=4.5, solid_capstyle="round")
        ax.plot([0], [y], marker="|", ms=9, color=INK)
    ax.set_yticks(range(len(cons)))
    ax.set_yticklabels([f"{r} ({d[:3]})" if d != "forward" else r
                        for r, d in zip(cons["reaction"][::-1], cons["direction"][::-1])], fontsize=7.5)
    ax.set_xlabel("activity, (value - median) / IQR")
    ax.set_title("Where the query asks each reaction to be")
    handles = [mpl.patches.Patch(color=c, label=m) for m, c in cols.items()] + [mpl.patches.Patch(color=GREY, label="diagnostic")]
    ax.legend(handles=handles, loc="lower right", fontsize=7.5)
    ax.grid(axis="y", visible=False)
    ax.set_xlim(np.percentile(Z, 1) - 0.5, np.percentile(Z, 99.5) + 0.8)
    return _save(fig, path)


def loop_removal(X_raw: np.ndarray, X_cf: np.ndarray, model, rxns: Sequence[str], path=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.3]})
    tot_r, tot_c = np.abs(X_raw).sum(1), np.abs(X_cf).sum(1)
    bins = np.linspace(min(tot_c.min(), tot_r.min()), max(tot_c.max(), tot_r.max()), 50)
    axes[0].hist(tot_r, bins=bins, color=C[1], alpha=0.75, label="uniform sampling")
    axes[0].hist(tot_c, bins=bins, color=C[0], alpha=0.75, label="after CycleFreeFlux")
    axes[0].set_xlabel("total flux  sum |v|  per sample")
    axes[0].set_ylabel("samples")
    axes[0].set_title("Most of the sampled flux runs in cycles")
    axes[0].legend()
    raw = [np.median(np.abs(X_raw[:, model.index(r)])) for r in rxns]
    cf = [np.median(np.abs(X_cf[:, model.index(r)])) for r in rxns]
    y = np.arange(len(rxns))
    axes[1].barh(y + 0.2, raw, height=0.38, color=C[1], label="uniform sampling")
    axes[1].barh(y - 0.2, cf, height=0.38, color=C[0], label="loop-free")
    axes[1].set_yticks(y)
    axes[1].set_yticklabels(rxns, fontsize=8)
    axes[1].set_xscale("symlog", linthresh=1)
    axes[1].set_xlabel("median |flux|  (umol/gDW/h, symlog)")
    axes[1].set_title("Requirement reactions before and after loop removal")
    axes[1].legend(loc="lower right")
    axes[1].invert_yaxis()
    fig.tight_layout()
    return _save(fig, path)


def sampling_compliance(frac: np.ndarray, mod: pd.DataFrame, path=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1, 1.4]})
    axes[0].hist(frac * 100, bins=np.linspace(0, 100, 26), color=C[0])
    axes[0].axvline(100, color=C[7], lw=1.5, ls="--")
    axes[0].set_xlabel("requirement groups met (%)")
    axes[0].set_ylabel("sampled states")
    axes[0].set_title(f"No sampled state is a growth state (best: {frac.max() * 100:.0f}%)")
    order = np.argsort(-mod.mean(axis=1).to_numpy())[:300]
    im = axes[1].imshow(mod.iloc[order].T.to_numpy(), aspect="auto", cmap=SEQ, vmin=0, vmax=1, interpolation="nearest")
    axes[1].set_yticks(range(mod.shape[1]))
    axes[1].set_yticklabels(mod.columns)
    axes[1].set_xlabel("300 most growth-like sampled states (ranked)")
    axes[1].set_title("Modules trade off against each other")
    axes[1].grid(False)
    fig.colorbar(im, ax=axes[1], label="module compliance", fraction=0.03)
    fig.tight_layout()
    return _save(fig, path)


def training_curves(hists: Dict[str, dict], path=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
    for i, (name, h) in enumerate(hists.items()):
        ep = np.arange(1, len(h["train_loss"]) + 1)
        axes[0].plot(ep, h["val_rec_mse"], color=C[i], label=f"{name} validation")
        axes[0].plot(ep, h["train_loss"], color=C[i], lw=1, ls=":", label=f"{name} training loss")
        if np.any(np.asarray(h["val_mass"]) > 0):
            axes[1].plot(ep, h["val_mass"], color=C[i], label=f"{name}")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("reconstruction MSE (scaled)")
    axes[0].set_title("Learning the loop-free flux space")
    axes[0].legend(fontsize=8)
    axes[1].set_yscale("log")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("normalised ||S v||^2 of decoded fluxes")
    axes[1].set_title("Physics penalty (PI-AE)")
    if axes[1].lines:
        axes[1].legend(fontsize=8)
    fig.tight_layout()
    return _save(fig, path)


def latent_map(Z_all: np.ndarray, score: np.ndarray, Z0: np.ndarray, Z1: np.ndarray, title: str, path=None):
    """PCA of the latent codes of the sampling coloured by soft score, with FluxQuery moves."""
    mu = Z_all.mean(0)
    _, _, Vt = np.linalg.svd(Z_all - mu, full_matrices=False)
    P = lambda Z: (Z - mu) @ Vt[:2].T
    A, B, Cc = P(Z_all), P(Z0), P(Z1)
    fig, ax = plt.subplots(figsize=(5.6, 4.4))
    o = np.argsort(score)
    sc = ax.scatter(A[o, 0], A[o, 1], c=score[o], s=4, cmap=SEQ, lw=0)
    for a, b in zip(B, Cc):
        ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="-|>", color=C[1], lw=1.1))
    ax.scatter(B[:, 0], B[:, 1], s=16, color="white", edgecolor=C[1], lw=1.2, label="seed", zorder=3)
    ax.scatter(Cc[:, 0], Cc[:, 1], s=16, color=C[1], edgecolor="white", lw=0.8, label="after FluxQuery", zorder=3)
    fig.colorbar(sc, ax=ax, label="soft score (growth-likeness)", fraction=0.04)
    ax.set_xlabel("latent PC1")
    ax.set_ylabel("latent PC2")
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return _save(fig, path)


def method_comparison(D: pd.DataFrame, order: List[str], path=None):
    metrics = [("compliance", "requirement groups met (%)", 100),
               ("nn_distance_percentile", "distance to sampled space\n(percentile of sample-sample NN)", 1),
               ("normalised_L1_change", "change from the seed\n(normalised L1)", 1)]
    fig, axes = plt.subplots(1, 3, figsize=(12, 0.32 * len(order) + 1.6), sharey=True)
    for ax, (col, lab, f) in zip(axes, metrics):
        for i, mth in enumerate(order):
            v = D.loc[D["method"] == mth, col].to_numpy(float) * f
            v = v[np.isfinite(v)]
            if not len(v):
                continue
            y = len(order) - 1 - i
            jit = (np.random.default_rng(i).random(len(v)) - 0.5) * 0.35
            ax.scatter(v, y + jit, s=10, color=C[i % len(C)], alpha=0.65, lw=0)
            ax.plot([np.median(v)] * 2, [y - 0.32, y + 0.32], color=INK, lw=2)
        ax.set_xlabel(lab)
        ax.grid(axis="y", visible=False)
    axes[0].set_yticks(range(len(order)))
    axes[0].set_yticklabels(order[::-1])
    axes[0].set_xlim(-3, 103)
    axes[1].set_xlim(-3, 103)
    axes[1].set_title("Which route finds typical, exact growth states?")
    fig.tight_layout()
    return _save(fig, path)


def module_heatmap(T: pd.DataFrame, title: str, path=None):
    fig, ax = plt.subplots(figsize=(0.95 * T.shape[1] + 2.4, 0.38 * T.shape[0] + 1.2))
    im = ax.imshow(T.to_numpy(float), cmap=SEQ, vmin=0, vmax=1, aspect="auto")
    for i in range(T.shape[0]):
        for j in range(T.shape[1]):
            v = T.iat[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if v > 0.6 else INK)
    ax.set_xticks(range(T.shape[1]))
    ax.set_xticklabels(T.columns, rotation=30, ha="right")
    ax.set_yticks(range(T.shape[0]))
    ax.set_yticklabels(T.index)
    ax.grid(False)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.03, label="compliance")
    return _save(fig, path)


def exchange_shift(dem: pd.DataFrame, path=None, top: int = 22):
    d = dem.dropna(subset=["cliffs_delta"]).copy()
    d = d[(d["uptake_growth_median"].abs() > 1e-6) | (d["uptake_background_median"].abs() > 1e-6)]
    d = d.reindex(d["cliffs_delta"].abs().sort_values(ascending=False).index)[:top]
    fig, ax = plt.subplots(figsize=(6.4, 0.27 * len(d) + 1.2))
    y = np.arange(len(d))[::-1]
    cols = [C[0] if v > 0 else C[1] for v in d["cliffs_delta"]]
    ax.barh(y, d["cliffs_delta"], color=cols, height=0.62)
    ax.axvline(0, color=INK2, lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(d["compound"])
    ax.set_xlim(-1.05, 1.05)
    ax.set_xlabel("Cliff's delta: uptake in growth states vs. background\n(+1 = always higher, -1 = always lower)")
    ax.set_title("Nutrients the growth phenotype consumes differently")
    ax.grid(axis="y", visible=False)
    return _save(fig, path)


def medium_pareto(scan: pd.DataFrame, names: Dict[str, str], chosen: float, path=None):
    fold_cols = [c for c in scan.columns if c.startswith("fold:") and (scan[c] > 1.01).any()]
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.5))
    axes[0].plot(scan["budget"], scan["violation"], marker="o", color=C[0], ms=5)
    axes[0].axvline(chosen, color=C[1], lw=1.2, ls="--", label="recommended")
    axes[0].set_xlabel("enrichment budget (sum of fold increases)")
    axes[0].set_ylabel("residual violation of the query")
    axes[0].set_title("How much medium change buys how much growth")
    axes[0].legend()
    for i, c in enumerate(fold_cols):
        axes[1].plot(scan["budget"], scan[c], marker="o", ms=4, color=C[i % len(C)], label=names.get(c[5:], c[5:]))
    axes[1].axvline(chosen, color=C[1], lw=1.2, ls="--")
    axes[1].set_xlabel("enrichment budget")
    axes[1].set_ylabel("fold change vs. base medium")
    axes[1].set_title("Which components the optimiser enriches")
    if fold_cols:
        axes[1].legend(fontsize=8)
    fig.tight_layout()
    return _save(fig, path)


def medium_formulation(form: pd.DataFrame, path=None):
    """Base vs recommended concentration of every component the design changes (log scale).
    Tier 1 (filled): availability-limited components, supported by the model as it stands.
    Tier 2 (open): additional changes if uptake of rate-limited components is not transport-saturated."""
    from matplotlib.lines import Line2D
    tier2 = form["fold_if_transport_unsaturated"] if "fold_if_transport_unsaturated" in form else pd.Series(1.0, form.index)
    d = form[((form["fold_change"] > 1) | (tier2 > 1)) & (form["base_uM"] > 0)].copy()
    if not len(d):
        return None
    d["t2"] = tier2.loc[d.index]
    d = d.sort_values(["fold_change", "t2"], ascending=True)
    fig, ax = plt.subplots(figsize=(7.2, 0.55 * len(d) + 1.6))
    y = np.arange(len(d))
    for yy, (_, r) in zip(y, d.iterrows()):
        flagged = bool(r.get("note", ""))
        col = C[1] if flagged else C[0]
        ax.plot([r["base_uM"], r["base_uM"] * max(r["fold_change"], r["t2"])], [yy, yy], color=GREY, lw=1.5, zorder=1)
        ax.scatter(r["base_uM"], yy, s=60, color=GREY, zorder=2)
        if r["fold_change"] > 1:
            ax.scatter(r["base_uM"] * r["fold_change"], yy, s=80, color=col, zorder=3)
            ax.text(r["base_uM"] * r["fold_change"] * 1.25, yy + 0.18, f"x{r['fold_change']:.1f}", fontsize=8, color=INK)
        if r["t2"] > 1 and abs(r["t2"] - r["fold_change"]) > 0.05:
            ax.scatter(r["base_uM"] * r["t2"], yy, s=80, facecolor="white", edgecolor=col, lw=1.8, zorder=3)
            ax.text(r["base_uM"] * r["t2"] * 1.25, yy - 0.32, f"x{r['t2']:.1f}", fontsize=8, color=INK2)
    lo = d["base_uM"].min() / 2
    hi = (d["base_uM"] * np.maximum(d["fold_change"], d["t2"])).max() * 4
    ax.set_xscale("log")
    ax.set_xlim(lo, hi)
    ax.set_ylim(-0.7, len(d) - 0.3)
    ax.set_yticks(y)
    ax.set_yticklabels([c + (" *" if n else "") for c, n in zip(d["compound"], d.get("note", pd.Series("", d.index)))])
    ax.set_xlabel("concentration (uM, log scale)")
    ax.set_title("Recommended growth medium: changes to the base medium")
    ax.grid(axis="y", visible=False)
    handles = [Line2D([], [], marker="o", ls="", color=GREY, label="base medium"),
               Line2D([], [], marker="o", ls="", color=C[0], label="growth medium (tier 1)"),
               Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor=C[0], mew=1.8,
                      label="if transport is not saturated (tier 2)")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.28 - 0.6 / len(d)), ncol=3, fontsize=8,
              frameon=False)
    if d.get("note", pd.Series("", d.index)).ne("").any():
        fig.text(0.01, 0.005, "* enriched through a known model artefact; see the note column", fontsize=7.5, color=INK2)
    fig.tight_layout()
    return _save(fig, path)


def objective_comparison(T: pd.DataFrame, path=None):
    """Compliance and distance to sampled space of classical objectives vs FluxQuery."""
    fig, ax = plt.subplots(figsize=(5.6, 4.0))
    markers = ["o", "s", "D", "^", "v", "P", "X", "*"]
    n = len(T)
    for i, (_, r) in enumerate(T.iterrows()):
        # later points are drawn smaller so that coincident results stay visible as concentric marks
        ax.scatter(r["nn_distance_percentile"], r["compliance"] * 100, s=150 - 90 * i / max(n - 1, 1),
                   marker=markers[i % len(markers)], color=C[i % len(C)], edgecolor="white", lw=1.2,
                   zorder=3 + i, label=r["method"])
    ax.set_xlabel("distance to sampled space (NN percentile)")
    ax.set_ylabel("growth requirements met (%)")
    ax.set_xlim(-3, 103)
    ax.set_ylim(-3, 103)
    ax.set_title("Classical objectives vs. the growth query")
    ax.legend(fontsize=7.5, loc="best")
    return _save(fig, path)

In [ ]:
style()
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4g}")
rng = np.random.default_rng(SEED)
print("FluxQuery", VERSION)

In [ ]:
MODEL_FILE = MODEL_FILE or select_file("Select the COBRA model (.mat)", ["*.mat"], env_var="FQ_MODEL")
SAMPLING_DIR = Path(SAMPLING_DIR or select_folder("Select the folder with the flux samplings", env_var="FQ_SAMPLES",
                                                  create=False))
QUERY_FILE = QUERY_FILE or select_file("Select the query document", ["*.docx", "*.xlsx", "*.csv", "*.yaml"], env_var="FQ_QUERY")
BIBLIOMIC_FILE = BIBLIOMIC_FILE or select_file("Select the bibliomic workbook (mediaData)", ["*.xlsx"], env_var="FQ_BIBLIOMIC")
OUTPUT_DIR = Path(OUTPUT_DIR or select_folder("Select (or create) the output folder", env_var="FQ_OUTPUT"))
FIG = OUTPUT_DIR / "figures"; FIG.mkdir(parents=True, exist_ok=True)
TAB = OUTPUT_DIR / "tables"; TAB.mkdir(parents=True, exist_ok=True)
for k, v in dict(model=MODEL_FILE, samplings=SAMPLING_DIR, query=QUERY_FILE, bibliomic=BIBLIOMIC_FILE,
                 output=OUTPUT_DIR).items():
    print(f"{k:>10}: {v}")

## 1. The model and its sampled flux space

A uniform sample of the feasible flux space is the raw material of everything that follows. The samplings are first converted from MATLAB to NumPy (once per folder), then pooled, and `MAX_SAMPLES` states are drawn at random. Before using them we check that every sampled state is a steady state within the model bounds.

In [ ]:
model = load_model(MODEL_FILE)
print(f"{model.name}: {model.n} reactions, {len(model.mets)} metabolites")
print(f"converting the samplings in {SAMPLING_DIR.name}/ (only new or changed files):")
SAMPLING_NPY = convert_samplings(SAMPLING_DIR, model)
X_raw = load_converted(SAMPLING_NPY, max_samples=MAX_SAMPLES, seed=SEED)
print(f"{len(X_raw):,} sampled states used, drawn from "
      f"{sum(len(np.load(p, mmap_mode='r')) for p in SAMPLING_NPY):,} in {len(SAMPLING_NPY)} file(s)")
pd.Series(sampling_validation(X_raw, model)).to_frame("sampling check")

### 1.1 Removing thermodynamically infeasible cycles

Uniform samplers explore every direction the stoichiometry allows, including internal cycles that run without any net conversion (for example a metabolite imported and exported through two transporters at once). Such cycles are thermodynamically impossible, but they are free in a constraint-based model, and in a high-dimensional polytope they dominate the volume. A reaction whose sampled percentiles are driven by cycle activity tells us nothing about biology.

We therefore project every sampled state onto its closest loop-free equivalent with **CycleFreeFlux** (Desouki *et al.*, Bioinformatics 2015). Exchange fluxes are fixed, every internal flux keeps its direction and can only shrink, and total internal flux is minimised. The nutrient budget of each state is untouched; only the cycling disappears. The projection is cached in the output folder so it is computed once per sampling.

In [ ]:
cache = OUTPUT_DIR / f"loopfree_samples_{len(X_raw)}.npy"
if not LOOP_FREE:
    X = X_raw
elif cache.exists() and np.load(cache, mmap_mode="r").shape == X_raw.shape:
    X = np.load(cache); print("loop-free sampling loaded from cache")
else:
    X = cycle_free_all(model, X_raw, every=2000).astype(np.float32)
    np.save(cache, X)
show = [r for r in ["ATPS4minew", "PGK", "PYK", "ICDHy", "GTHO", "G6PDH2c", "GLNt4", "SERt4", "GLCt1r", "ATPM",
                    "CEPTC", "PSDm_hs"] if model.has(r)]
loop_removal(X_raw, X, model, show, path=FIG / "01_loop_removal.png"); plt.show()
print(f"median total flux per state: {np.median(np.abs(X_raw).sum(1)):,.0f} -> {np.median(np.abs(X).sum(1)):,.0f} "
      f"(exchange fluxes unchanged: max |diff| = {np.abs((X_raw - X)[:, model.exchanges()]).max():.2g})")

In the uniform sampling about two thirds of all flux circulates in cycles. Transporters such as `GLNt4` carry thousands of units, while net glutamine uptake is only tens, and several "energy" reactions sit near the generic bound of 10,000. After loop removal the exchange fluxes are identical, but internal fluxes now reflect what the cell actually does with its nutrients.

## 2. The query: what does axonal growth require?

The query document lists the requirements by module (energy, membrane lipids, polyamines, redox, nutrient uptake) plus a few identity reactions for each neuronal type. Each row gives a direction and a percentile window. The notebook re-checks every row against this model and its loop-free sampling and demotes the rows it cannot use (missing, blocked, opposite sign, extreme flux near the generic bound). Perfectly coupled rows are merged into one group so they count once.

> **About the query.** Version 4 corrects the original v3.1 table. `ACCAOC` (acetyl-CoA carboxylase) feeds fatty-acid synthesis, not cholesterol. Putrescine acetylation is polyamine *catabolism*. Nutrient uptakes were measured on internal transporters that carry cycles; they are now measured on net exchange reactions. Serine is net *secreted* by these neurons. Section 4 of the document lists every change with its justification.

In [ ]:
query_all = read_query(QUERY_FILE)
print("contexts in the query:", available_contexts(query_all))
query = select_context(query_all, CONTEXT)
report = validate_query(query, model, X)
query_used = apply_validation(query, report)
cons = build_constraints(query_used, model, X)
act = active(cons)
print(f"{len(query)} requirement rows for '{CONTEXT}': {len(act)} active ({act['group'].nunique()} independent groups), "
      f"{(cons['status'] != 'active').sum()} diagnostic, {(~query['reaction'].map(model.has)).sum()} not in the model")
report.to_csv(TAB / "query_validation.csv", index=False)
cols = ["reaction", "module", "direction", "status", "P5", "P50", "P95", "verdict", "coupled_with"]
report.loc[report["verdict"].ne("ok") | report["coupled_with"].ne(""), cols]

In [ ]:
query_thresholds(X, cons, path=FIG / "02_query_thresholds.png"); plt.show()
cons[["module", "reaction", "direction", "status", "group", "p_low", "p_high", "target_lb", "target_ub"]].to_csv(
    TAB / "query_thresholds.csv", index=False)

### 2.1 Does the sampled space already contain growth states?

In [ ]:
frac, viol, soft = compliance(X, cons)
mod_X = module_compliance(X, cons)
sampling_compliance(frac, mod_X, path=FIG / "03_sampling_compliance.png"); plt.show()
print(f"best sampled state meets {frac.max():.0%} of the requirement groups; median {np.median(frac):.0%}; "
      f"states meeting all: {(frac >= 1).sum()}")
mod_X.describe().T[["mean", "50%", "max"]].rename(columns={"50%": "median"})

No sampled state satisfies the whole query, and the best ones satisfy about half of it. The modules compete: states rich in membrane synthesis are poor in energy or nutrient uptake, and vice versa. A growth state is a rare, coordinated corner of the flux space. Sampling alone will not find it, which is the reason for a directed search.

## 3. Learning the flux space

Following the original FluxQuery design, both autoencoders are trained on **all** sampled states. The phenotype is not used to filter the training data; it only enters afterwards, as the question asked of the trained model.

* **AE**: dense encoder 512-128-latent and a mirrored decoder; reconstruction loss 0.8·MSE + 0.2·Huber in signed-log, robust-scaled flux space, plus a light L1 penalty on the latent code.
* **PI-AE**: the same network, with the decoded fluxes additionally penalised for violating mass balance (‖S v‖²) and the model bounds.

The networks are implemented in plain NumPy (explicit back-propagation, AdamW, one-cycle learning rate), so the notebook needs no deep-learning framework or GPU.

In [ ]:
pre = Preprocessor.fit(X)
Xs = pre.transform(X)
physics = Physics.from_model(model, X)
nets, hists = {}, {}
for name, lam_m, lam_b in [("AE", 0.0, 0.0), ("PI-AE", PI_LAMBDA_MASS, PI_LAMBDA_BOUNDS)]:
    print(f"--- {name} ---")
    nets[name] = Autoencoder(model.n, pre, latent=LATENT_DIM, seed=SEED)
    cfg = TrainConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE, lambda_mass=lam_m,
                        lambda_bounds=lam_b, seed=SEED, verbose_every=20)
    hists[name] = train(nets[name], Xs, cfg, physics)
training_curves(hists, path=FIG / "04_training.png"); plt.show()
val = hists["AE"]["val_index"]
rec = pd.DataFrame({k: reconstruction_report(nets[k], Xs[val], X[val], model) for k in nets}).T
rec["best_epoch"] = [hists[k]["best_epoch"] for k in nets]
rec["minutes"] = [hists[k]["seconds"] / 60 for k in nets]
rec.to_csv(TAB / "reconstruction.csv"); rec

In [ ]:
# linear baseline: PCA fitted on the same training states, evaluated on the same validation states
tr_idx = hists["AE"]["train_index"]
mu = Xs[tr_idx].mean(0)
_, sv, Vt = np.linalg.svd(Xs[tr_idx] - mu, full_matrices=False)
ev = np.cumsum(sv ** 2) / np.sum(sv ** 2)
print("variance of the loop-free flux space captured by the first k principal components:",
      {k: f"{ev[k - 1]:.1%}" for k in (8, 16, 32, 64, 128, 256)})
Xv = Xs[val] - mu
for k in (64, LATENT_DIM):
    R = Xv @ Vt[:k].T @ Vt[:k]
    print(f"PCA with {k} components, validation R2 in the same scaled space: "
          f"{1 - ((Xv - R) ** 2).sum() / (Xv ** 2).sum():.3f}")

Three observations, which matter for how much weight the network results deserve:

* **The loop-free flux space is low-dimensional.** About 64 linear directions carry roughly 90% of its variance, and 128 nearly all of it. Cycles are what made the raw sampling look high-dimensional.
* **The physics-informed network trades reconstruction for physics.** Its decoded fluxes have a steady-state residual about five times lower than the plain AE, and a lower absolute flux error, but a lower R² in the scaled space.
* **With 8,000 states, linear PCA reconstructs the space better than either network.** The networks overfit: training loss keeps falling while validation loss flattens. This is expected with 5,600 training states for a 1,915-dimensional input and about two million weights. With the full sampling (roughly 240,000 states in `samples1-3.mat`) this comparison should be repeated. If PCA still wins, the honest conclusion is that a linear latent space is enough for this model.

## 4. FluxQuery: searching the learned space for growth states

The search freezes the network and moves only the latent codes of a set of seeds. The seeds are drawn at random from the 5% most growth-like sampled states, so that the result is an ensemble rather than a single answer. The objective rewards meeting the query, keeps the latent code close to where it started, and softly penalises mass imbalance and bound violations; the physics terms are auto-scaled to 10% of the biological term at the start.

The search returns candidates in two forms:

* **raw**: the decoded flux vector after the search (the original formulation);
* **residual**: the seed plus the *change* the network proposes (decoded after − decoded before). Decoders blur fine detail, so a decoded vector is never exactly a sampled state; taking the difference cancels that bias and keeps the seed's detail.

Every candidate is then made an exact steady state by linear programming, in one of two ways:

* **L1 repair** (original code): the nearest state that is exactly feasible.
* **bio-LP repair** (new): the nearest exactly feasible state *that meets the query*, with biology weighted 100 times more than distance.

Two references without a network bracket the comparison:

* **seed + bio-LP**: the same LP applied directly to the sampled seed.
* **LP oracle**: the best possible query compliance, ignoring how typical the state is.

In [ ]:
seeds = choose_seeds(X, cons, N_SEEDS, seed=SEED)
runs = {}
for name, net in nets.items():
    print(f"--- FluxQuery on {name} ---")
    runs[name] = latent_search(net, physics, cons, Xs[seeds], steps=FQ_STEPS, lr=FQ_LR, lambda_bio=FQ_LAMBDA_BIO,
                        lambda_latent=FQ_LAMBDA_LATENT, log_every=200)
Z_all = np.vstack([nets["PI-AE"].encode(Xs[i:i + 4096]) for i in range(0, len(Xs), 4096)])
latent_map(Z_all, soft, runs["PI-AE"]["Z0"], runs["PI-AE"]["Z"], "PI-AE latent space: FluxQuery moves seeds towards growth",
             path=FIG / "05_latent_map.png"); plt.show()

In [ ]:
prox = Proximity(pre, X, seed=SEED)
scale = flux_scale(X)
msign = np.sign(np.median(X, axis=0))
rows, candidates = [], []

def add(method, v, start):
    s = summarise(method, v, cons, model, prox, start=start, scale=scale)
    rows.append(s); candidates.append((method, v))

t0 = time.time()
for k, i in enumerate(seeds):
    x = X[i].astype(float)
    rows.append(summarise("sampled seed", x, cons, model, prox))
    add("seed + bio-LP", query_lp(model, cons, v0=x, scale=scale, rho=100, median_sign=msign)["v"], x)
    for name, r in runs.items():
        raw = r["V_after"][k].astype(float)
        res = x + (r["V_after"][k] - r["V_before"][k]).astype(float)
        rows.append(summarise(f"{name} raw", raw, cons, model, prox))
        add(f"{name} raw + L1 repair", nearest_feasible(model, raw, scale), x)
        add(f"{name} raw + bio-LP", query_lp(model, cons, v0=raw, scale=scale, rho=100, median_sign=msign)["v"], x)
        add(f"{name} residual + bio-LP", query_lp(model, cons, v0=res, scale=scale, rho=100, median_sign=msign)["v"], x)
oracle = query_lp(model, cons, median_sign=msign)["v"]
rows.append(summarise("LP oracle", oracle, cons, model, prox))
print(f"{len(candidates)} exact candidates in {time.time() - t0:.0f} s")
D = pd.concat(rows, ignore_index=True)
order = ["sampled seed", "seed + bio-LP"] + [f"{n} {s}" for n in nets for s in ["raw", "raw + L1 repair", "raw + bio-LP", "residual + bio-LP"]] + ["LP oracle"]
method_comparison(D, order, path=FIG / "06_method_comparison.png"); plt.show()
summary = D.groupby("method", sort=False)[["compliance", "soft_score", "max_abs_Sv", "nn_distance_percentile",
                                           "normalised_L1_change"]].median().reindex(order)
summary.to_csv(TAB / "method_comparison.csv"); summary

How to read this comparison:

* **Raw network outputs** are not steady states: their mass-balance residuals are large, as expected from any decoder. The physics-informed network reduces them by about half.
* **Plain L1 repair**, as in the original code, restores feasibility but throws away most of what the search gained: compliance falls back to the level of an ordinary sampled state.
* **Biology-aware repair** reaches 100% compliance from every starting point. The routes differ in how far they travel and how typical the result is.
* **The LP oracle** proves that the query is jointly satisfiable in this model. Its single answer, however, is an extreme vertex of the polytope, far from anything the sampler ever visited.

The fair question is whether the network finds growth states that are both exact and *typical* of the model's behaviour. In the validation run the answer is mixed, and it is worth stating plainly:

* **PI-AE residual + bio-LP** gives fully compliant states that sit inside the sampled cloud (median nearest-neighbour percentile about 3) with small changes from the seed.
* **seed + bio-LP** does just as well without any network. On 8,000 states, the decisive ingredient is the biology-aware LP, not the latent search.
* **The plain AE search** improves the query score the most in latent space, but it drifts away from the sampled manifold (percentile above 80). The physics-informed search stays on the manifold but moves little.

The ensemble used downstream pools every fully compliant candidate whose distance to the sampled space is not larger than that of a typical sampled state (percentile ≤ 50), whatever route produced it. Loop removal is then applied with the requirement reactions held fixed, so the ensemble stays 100% compliant.

In [ ]:
V_cand = np.array([v for _, v in candidates])
meta = pd.concat([summarise(m, v, cons, model, prox) for m, v in candidates], ignore_index=True)
keep = (meta["compliance"] >= 1 - 1e-9) & (meta["nn_distance_percentile"] <= 50)
query_idx = act["rxn_index"].to_numpy(int)          # requirement reactions stay fixed during loop removal
V_growth = np.array([cycle_free(model, v, fixed=query_idx) for v in V_cand[keep.to_numpy()]])
print(f"growth ensemble: {len(V_growth)} exact, loop-free states; "
      f"compliance after loop removal: {compliance(V_growth, cons)[0].mean():.0%}")
print(meta.loc[keep, "method"].value_counts().to_string())
mod_table = pd.DataFrame({
    "sampled states (all)": module_compliance(X, cons).mean(),
    "best sampled seeds": module_compliance(X[seeds], cons).mean(),
    **{f"{n} raw": module_compliance(runs[n]["V_after"], cons).mean() for n in runs},
    "growth ensemble": module_compliance(V_growth, cons).mean(),
}).T
module_heatmap(mod_table, "Module compliance along the pipeline", path=FIG / "07_module_compliance.png"); plt.show()

## 5. Comparison with classical objective functions

Constraint-based studies of neurons usually pick one objective: maximise ATP demand, or a product of interest. We solve the same model with several such objectives (parsimonious FBA, so each answer is unique) and score the answers against the growth query.

In [ ]:
objs = default_objectives(model, CONTEXT)
V_obj, obj_tab = run_objectives(model, objs)
comp = pd.concat([summarise(m, v, cons, model, prox) for m, v in zip(obj_tab["method"], V_obj)] +
                 [summarise("FluxQuery growth ensemble (median)", V_growth, cons, model, prox).median(numeric_only=True)
                  .to_frame().T.assign(method="FluxQuery growth ensemble (median)")], ignore_index=True)
mod_obj = pd.concat([module_compliance(V_obj, cons).set_index(obj_tab["method"]),
                     module_compliance(V_growth, cons).mean().to_frame("FluxQuery growth ensemble").T])
objective_comparison(comp, path=FIG / "08_objectives_scatter.png"); plt.show()
module_heatmap(mod_obj, "What each objective delivers, module by module", path=FIG / "09_objectives_modules.png"); plt.show()
key = [r for r in ["ATPM", "EX_glc_D[e]", "EX_o2[e]", "EX_lac_L[e]", "PDHm", "CEPTC", "PSDm_hs", "SMS", "MEVK1c",
                   "ORNDC", "GTHS", "G6PDH2c"] if model.has(r)]
flux_cmp = pd.DataFrame(V_obj[:, [model.index(r) for r in key]], columns=key, index=obj_tab["method"])
flux_cmp.loc["FluxQuery growth ensemble (median)"] = np.median(V_growth[:, [model.index(r) for r in key]], axis=0)
flux_cmp.loc["loop-free sampling (median)"] = np.median(X[:, [model.index(r) for r in key]], axis=0)
comp.to_csv(TAB / "objective_comparison.csv", index=False); flux_cmp.to_csv(TAB / "objective_fluxes.csv")
comp[["method", "compliance", "soft_score", "nn_distance_percentile"]]

In [ ]:
flux_cmp

Each classical objective drives the cell to one extreme. Maximising ATP demand saturates oxidative metabolism at the expense of biosynthesis. Maximising lipid synthesis does the opposite. Parsimonious "minimal" states do almost nothing. None of them meets the coordinated set of requirements that axonal growth implies, and all of them lie far outside the sampled space. The growth query instead describes a balanced phenotype: high, but not maximal, ATP turnover together with active membrane, polyamine and glutathione synthesis.

## 6. From growth states to a culture medium

### 6.1 A medium-consistent model

Exchange reactions are the interface between the cell and the medium. Before translating fluxes into concentrations we must make sure that the model takes up only what the medium contains. Genome-scale models often keep uptake reactions open for metabolites that are not in the medium (to stay feasible during reconstruction). Left open, these uptakes become phantom nutrients: in iDopaNeuroC the sampled states import galactose, ATP, L-DOPA and hydrogen peroxide in large amounts.

The medium stage therefore closes every uptake of a metabolite absent from the medium. It keeps water, protons, O₂/CO₂, measured uptakes, and the minimal uptakes the network cannot do without (found by linear programming and reported). The growth ensemble is then re-projected into this medium-consistent model with the same biology-aware LP.

In [ ]:
medium = Medium.from_bibliomic(BIBLIOMIC_FILE)
lb_m, ub_m, closed = consistent_bounds(model, medium)
print(f"conversion: 1 uM in the medium sustains {medium.k:.4f} umol/gDW/h over one culture interval")
print(f"{len(closed)} non-medium uptakes closed; kept for feasibility: "
      f"{closed.loc[closed['new_lb'] < 0, ['exchange', 'new_lb']].to_dict('records')}")
art = closed.assign(median_uptake_in_sampling=[-np.median(X[:, model.index(r)]) for r in closed["exchange"]])
art = art[art["median_uptake_in_sampling"] > 1].sort_values("median_uptake_in_sampling", ascending=False)
art.to_csv(TAB / "medium_artefactual_uptakes.csv", index=False)
art.head(12)

In [ ]:
ref_m = query_lp(model, cons, lb=lb_m, ub=ub_m, median_sign=msign)
rr = row_report(ref_m["v"], active(cons))
print(f"best achievable compliance in the medium-consistent model: {compliance(ref_m['v'], cons)[0][0]:.0%}")
unreachable = rr.loc[~rr["met"], ["module", "reaction", "target_lb", "activity", "normalised_violation"]]
unreachable

With phantom nutrients removed, some thresholds can no longer be reached, because they were learnt from a sampling that relied on those uptakes:

* **Aromatic L-amino-acid decarboxylase (`3HLYTCL`)** was fed by imported L-DOPA. From tyrosine alone, at the measured uptake rate, it cannot approach the sampled percentile.
* **BH4 synthesis (`GTPCI`/`SPR`)** was supported by phantom carbon and nucleotides.
* **Glutathione synthesis (`GTHS`)** falls from a sampled threshold of about 420 to a maximum of about 17. Here the medium, not the model, is the bottleneck, and the medium design below targets it.

In [ ]:
def to_medium(V):
    out = []
    for v in V:
        try:
            # biology weighted 10x more than in Section 4: closing phantom uptakes moves many states far
            out.append(query_lp(model, cons, lb=lb_m, ub=ub_m, v0=v, scale=scale, rho=1000, median_sign=msign)["v"])
        except LPError:
            pass
    return np.array(out)

V_growth_m = np.array([cycle_free(model, v, lb_m, ub_m, fixed=query_idx) for v in to_medium(V_growth)])
bg_idx = rng.choice(len(X), min(200, len(X)), replace=False)
V_bg_m = np.array([nearest_feasible(model, X[i].astype(float), scale, lb_m, ub_m) for i in bg_idx])
print(f"medium-consistent growth states: {len(V_growth_m)}; background states: {len(V_bg_m)}")
print(f"compliance of growth states in the medium-consistent model: {compliance(V_growth_m, cons)[0].mean():.0%} "
      f"(background: {compliance(V_bg_m, cons)[0].mean():.0%})")
demand = demand_table(model, medium, V_growth_m, V_bg_m, lb_m)
demand.to_csv(TAB / "nutrient_demand.csv", index=False)
exchange_shift(demand, path=FIG / "10_nutrient_demand.png"); plt.show()
demand.loc[demand["in_model"], ["compound", "base_uM", "limitation", "uptake_growth_median", "uptake_background_median",
                                "cliffs_delta", "fraction_of_availability_growth"]].sort_values("cliffs_delta", ascending=False)

### 6.2 Designing the medium

Medium components fall into two classes:

* **Availability-limited.** The cell can take up everything the medium offers in one culture interval, so the medium concentration *is* the constraint and raising it raises the uptake capacity.
* **Rate-limited.** The model bound comes from a measured consumption rate far below what the medium offers, so a higher concentration changes nothing unless the transporter is unsaturated.

The design LP lets the concentration of every availability-limited component rise (up to `MAX_FOLD`), under a total "enrichment budget" (the sum of fold increases), and minimises the residual violation of the growth query. Scanning the budget shows how much medium change buys how much growth. We recommend the budget at the knee of that curve: the largest budget at which one more unit of enrichment still buys at least a quarter of what the first unit bought. A sensitivity run also allows rate-limited components to scale, under the explicit hypothesis of unsaturated transport.

In [ ]:
classes = classify_components(model, medium, lb_m)
enrich = classes.loc[classes["limitation"] == "availability-limited", "rxn"].tolist()
names = dict(zip(medium.table["rxn"], medium.table["compound"]))
budgets = (0, 0.5, 1, 2, 4, 8, 16, 32)
scan = budget_scan(model, cons, medium, lb_m, ub_m, enrich, budgets=budgets, max_fold=MAX_FOLD, median_sign=msign)
chosen = choose_budget(scan, slope_fraction=0.25)
medium_pareto(scan, names, chosen, path=FIG / "11_medium_pareto.png"); plt.show()
print(f"recommended enrichment budget: {chosen}")
scan[["budget", "violation", "compliance"] + [c for c in scan.columns if c.startswith("fold:") and (scan[c] > 1.01).any()]]

In [ ]:
rate = classes.loc[classes["limitation"].str.startswith("rate"), "rxn"].tolist()
scan_rate = budget_scan(model, cons, medium, lb_m, ub_m, enrich + rate, budgets=budgets, max_fold=MAX_FOLD, median_sign=msign)
medium_pareto(scan_rate, names, chosen, path=FIG / "12_medium_pareto_transport_hypothesis.png"); plt.show()
scan_rate[["budget", "violation", "compliance"] + [c for c in scan_rate.columns if c.startswith("fold:") and (scan_rate[c] > 1.01).any()]]

In [ ]:
candidates_supp = {"EX_creat[e]": "Creatine", "EX_chsterol[e]": "Cholesterol", "EX_34dhphe[e]": "L-DOPA",
                   "EX_arachd[e]": "Arachidonic acid", "EX_lnlncg[e]": "gamma-Linolenic acid", "EX_gal[e]": "D-Galactose",
                   "EX_ac[e]": "Acetate", "EX_lac_L[e]": "L-Lactate", "EX_bhb[e]": "beta-Hydroxybutyrate",
                   "EX_cit[e]": "Citrate", "EX_orn[e]": "L-Ornithine", "EX_ser_L[e]": "L-Serine",
                   "EX_uri[e]": "Uridine", "EX_crvnc[e]": "Docosahexaenoic acid", "EX_spmd[e]": "Spermidine"}
# components the LP would enrich only through a known model artefact are excluded from the final design
artefact = [r for r in enrich if r in BIOLOGICAL_CAVEATS]
enrich_bio = [r for r in enrich if r not in BIOLOGICAL_CAVEATS]
best_all = design_lp(model, cons, medium, lb_m, ub_m, enrich, chosen, max_fold=MAX_FOLD, median_sign=msign)
best = design_lp(model, cons, medium, lb_m, ub_m, enrich_bio, chosen, max_fold=MAX_FOLD, median_sign=msign)
print(f"excluded as model artefacts: {[names.get(r, r) for r in artefact]}; residual violation "
      f"{best_all['violation']:.3g} (all components) vs {best['violation']:.3g} (artefacts excluded)")
supp = supplement_screen(model, cons, medium, lb_m, ub_m, candidates_supp, dose_uM=SUPPLEMENT_DOSE_UM,
                           enrichable=enrich_bio, budget=chosen, max_fold=MAX_FOLD, median_sign=msign)
supp.to_csv(TAB / "supplement_screen.csv", index=False)
supp.sort_values("violation_reduction_%", ascending=False, na_position="last")

In [ ]:
rate_bio = [r for r in rate if r not in BIOLOGICAL_CAVEATS]
best_rate = design_lp(model, cons, medium, lb_m, ub_m, enrich_bio + rate_bio, chosen, max_fold=MAX_FOLD, median_sign=msign)
form = formulation(medium, classes, best["fold"], demand, supp, fold_transport=best_rate["fold"])
form.to_csv(TAB / "medium_recommendation.csv", index=False)
medium_formulation(form, path=FIG / "13_medium_recommendation.png"); plt.show()
changed = form[(form["fold_change"] > 1) | (form["fold_if_transport_unsaturated"] > 1) | form["group"].eq("supplement")]
changed[["compound", "group", "base_uM", "recommended_uM", "fold_change", "uM_if_transport_unsaturated",
         "limitation", "recommendation", "note"]]

### 6.3 Does the new medium favour growth?

A medium cannot force a cell into a state, but it decides which states are possible. We compare the base and the recommended medium in three ways:

* the best achievable compliance with the growth query;
* the maximum flux each growth reaction can carry in each medium (flux variability analysis), i.e. the capacity the medium makes available;
* the per-requirement residuals.

In [ ]:
def medium_bounds(fold):
    lb2 = lb_m.copy()
    for r, f in fold.items():
        lb2[model.index(r)] = lb_m[model.index(r)] * f
    return lb2

lb_new = medium_bounds(best["fold"])
check = {}
for label, lbx in [("base medium", lb_m), ("growth medium", lb_new)]:
    sol = query_lp(model, cons, lb=lbx, ub=ub_m, median_sign=msign)
    check[label] = {"best compliance": compliance(sol["v"], cons)[0][0], "residual violation": sol["bio_violation"]}
growth_rxns = [r for r in ["GLUCYS", "GTHS", "CEPTC", "PSDm_hs", "SMS", "MEVK1c", "ORNDC", "SPRMS", "G6PDH2c", "ATPM", "PDHm"]
               if model.has(r)]
fva_tab = pd.concat({k: fva(model, growth_rxns, lbx, ub_m).set_index("reaction")
                 for k, lbx in [("base medium", lb_m), ("growth medium", lb_new)]}, axis=1)
fva_tab.to_csv(TAB / "fva_base_vs_growth_medium.csv")
display(pd.DataFrame(check).T)
cap = pd.DataFrame({"max flux, base medium": fva_tab[("base medium", "max")],
                    "max flux, growth medium": fva_tab[("growth medium", "max")]})
cap["fold"] = cap["max flux, growth medium"] / cap["max flux, base medium"].where(cap["max flux, base medium"].abs() > 1e-9)
cap

## 7. Interpretation

Numbers below are from the validation run: iDopaNeuroC, 8,000 sampled states, 37 independent requirement groups. Each point should be checked again on the production sampling.

### What a growing axon looks like, metabolically

Growth is not "more of everything". Compared with the median loop-free sampled state, the growth ensemble has:

* about 1.5× the ATP turnover, but far below the maximum the model allows;
* pyruvate oxidation (PDH) about 1.4× higher, with O₂ uptake essentially unchanged and lactate release slightly higher, so oxidative and glycolytic ATP supply work together, as reported for growing and regenerating axons;
* at the same time, phosphatidylcholine synthesis (CEPTC) about 5× higher, the mevalonate pathway about 3×, ornithine decarboxylase about 2×, and NADPH production through the pentose-phosphate pathway and glutathione synthesis about 1.6× and 1.3×.

No sampled state satisfies this combination (the best meets about half of it), and no classical objective comes close:

* Maximising ATP demand meets 5% of the requirements.
* Maximising membrane-lipid synthesis meets 30%.
* NADPH, minimal-flux and dopamine-release objectives meet 0–14%.
* All of them sit at the edge of the flux space, beyond 99% of sampled states.

The growth query describes a *balanced* phenotype. That is the main argument for querying a flux space instead of optimising a single objective.

### What the network contributes

Loop removal is the step that made learning possible. About two thirds of the sampled flux was cycling, and without removing it the autoencoders were learning cycles.

After loop removal:

* the physics-informed network produces decoded fluxes close to steady state;
* its search, followed by biology-aware LP repair, yields growth states that are exact and typical of the model.

Two caveats from this run:

* A network-free route (sampled seed + the same LP) is equally good.
* Linear PCA reconstructs the 8,000-state space better than both networks.

The original L1 repair was the weak point of v3.2: it restored feasibility by undoing the biology (compliance 3–14%). The biology-aware repair fixes that. Whether the network adds value beyond the LP should be decided on the full sampling, where it has 30 times more data.

### The medium

Closing 31 phantom uptakes changes the picture: galactose, H₂O₂, DOPAL, ATP, L-DOPA and others were imported at thousands of units in the sampling. In the medium-consistent model, the growth states still meet 92% of the requirements; background states meet 17%.

**Tier 1, supported by the model as it stands: raise L-cystine about 9-fold (47 → 424 µM).**

* Cystine is the only availability-limited component the design LP enriches before the knee of the budget curve.
* Growth states consume all the cystine the medium offers.
* Glutathione synthesis is the requirement furthest from its target. Its maximum capacity rises from 17 to about 140 with the enriched medium.

This matches the biology: neuronal glutathione depends on extracellular cyst(e)ine (Dringen 2000), and glutathione protects growth cones from the oxidative load of high oxidative ATP production and lipid synthesis.

**Tier 2, if uptake is not saturated by transport: raise glutamine about 2.8-fold, then glucose.** These are the two main fuels of growth. The model cannot decide whether their measured uptake rates are transport-limited, so this tier is a hypothesis to test.

**What the model does not support.** Creatine, lipid precursors, cholesterol and galactose do not move the query, because membrane synthesis in this model is limited by carbon and NADPH, not by head groups or precursors. Phenylalanine is enriched by the LP only through hepatic phenylalanine hydroxylase, a model artefact, so it was excluded.

### Practical notes for the bench

* **Cystine at 424 µM is close to its solubility at neutral pH (~0.45 mM).** Prepare it from an acidic or alkaline stock, or titrate it (100, 200, 400 µM).
* **Neurons take up cysteine mainly through EAAT3/EAAC1 and rely on astrocytes for cystine** (Aoyama et al. 2006). In pure neuronal cultures, N-acetylcysteine is a practical cysteine donor to test alongside cystine.
* **High cystine increases glutamate release through system xc⁻** where it is present. Glutamine above about 2 mM has the same caveat for excitotoxicity, which is why tier 2 is a hypothesis.
* **Readouts:** axon length (Tau-1 or βIII-tubulin), intracellular GSH, and oxygen consumption rate as a check of the predicted oxidative shift.

### Limitations

* Thresholds are relative to the sampled space. The sampling of iDopaNeuroC was generated with phantom uptakes open, so a few thresholds (DOPA decarboxylation from imported L-DOPA, BH4 synthesis) cannot be reached in the medium-consistent model. **Re-sampling the model with the medium-consistent bounds exported below is the most important next step.**
* Uptake capacities assume the medium is consumed over one culture interval (48 h in 2 mL). Components whose uptake is limited by a measured rate are not enriched in tier 1.
* The model has no membrane-accretion reaction. Membrane synthesis is read from pathway fluxes, not from net membrane growth.

In [ ]:
pd.DataFrame({"lb": lb_m, "ub": ub_m}, index=model.rxns).to_csv(TAB / "medium_consistent_bounds.csv")
pd.DataFrame(V_growth_m, columns=model.rxns).to_csv(TAB / "growth_ensemble_medium_consistent.csv", index=False)
pd.DataFrame(V_growth, columns=model.rxns).to_csv(TAB / "growth_ensemble.csv", index=False)
with pd.ExcelWriter(OUTPUT_DIR / f"medium_recommendation_{CONTEXT}.xlsx") as xw:
    form.to_excel(xw, sheet_name="recommended medium", index=False)
    scan.to_excel(xw, sheet_name="budget scan", index=False)
    scan_rate.to_excel(xw, sheet_name="budget scan (transport hyp.)", index=False)
    supp.to_excel(xw, sheet_name="supplement screen", index=False)
    demand.to_excel(xw, sheet_name="nutrient demand", index=False)
    closed.to_excel(xw, sheet_name="closed uptakes", index=False)
meta_run = {"context": CONTEXT, "model": Path(MODEL_FILE).name, "sampling": [p.name for p in SAMPLING_NPY],
            "n_states": int(len(X)), "loop_free": LOOP_FREE, "query": Path(QUERY_FILE).name,
            "active_requirements": int(len(act)), "groups": int(act["group"].nunique()),
            "growth_ensemble": int(len(V_growth)), "recommended_budget": chosen,
            "epochs": EPOCHS, "latent_dim": LATENT_DIM, "seeds": int(N_SEEDS), "fluxquery_version": VERSION}
json.dump(meta_run, open(OUTPUT_DIR / "run_metadata.json", "w"), indent=2)
print("written to", OUTPUT_DIR); sorted(p.name for p in OUTPUT_DIR.rglob("*") if p.is_file())

## 8. Re-using the notebook for other neurons

1. Build the cell-type model and its flux sampling (same pipeline as iDopaNeuroC) and put the sampling files in their own folder. The curated bibliomic data for glutamatergic and cholinergic neurons are Supplementary Data S2 and S1.
2. Set `CONTEXT = "glutamatergic"` or `"cholinergic"`. The query already contains their identity rows: vesicular glutamate release, glutaminase and NAA synthesis; or choline acetyltransferase, vesicular acetylcholine release, high-affinity choline uptake and ATP-citrate lyase.
3. Run the notebook. Section 2 reports any requirement that does not transfer to the new model, and Section 6 any phantom uptake of that model.